# SignSage T5-small Pose-to-Text POC (Kaggle GPU)

**Self-contained: the bootstrap cell below reconstructs every required source file
from blobs embedded in this notebook (exact bytes of commit `981dd82`). No `git clone`,
no repository checkout, and no network access are needed for bootstrap.**

Set `EXECUTE = True` in the config cell, then use Run-All. Phase cells still enforce
hard inter-phase gates via `state.json`:

- **PHASE 3 FAIL (>=95% EM not met) -> STOP.** Phase 4 will refuse to run.
- Any failed `subprocess` (`check=True`) halts the notebook immediately.

Flow: Bootstrap -> Phase 0 setup -> Phase 1 smoke -> Phase 2 materialize ->
Phase 3 gate -> Phase 4 5k -> Phase 5 evaluate -> Phase 6 publish.


In [ ]:
# CONFIG — edit these, then set EXECUTE = True and Run-All.
from pathlib import Path
EXECUTE = False  # set True only when you intentionally want execution
POC = Path("/kaggle/working/t5_poc")   # canonical runtime root (code + outputs)
# SUP must equal materialize.py's REPO_ROOT (= file.parents[2]). Since POC is the
# directory CONTAINING materialize.py (one level above __file__), SUP = POC.parents[1].
# (POC.parents[2] would point one level too high — that was the Kaggle bootstrap bug.)
SUP = POC.parents[1]                     # support root: hosts backend/, research/
OUT = POC                                # smoke/ shards/ gate64/ exp5k/ eval5k/ reports/ live here
CSV = ""  # Phase 2 input, e.g. "/kaggle/input/isign/isign_v1.1.csv"
print(f"EXECUTE={EXECUTE} POC={POC} SUP={SUP}")

import json as _json
def set_state(phase, status):
    OUT.mkdir(parents=True, exist_ok=True)
    sf = OUT / "state.json"
    st = _json.loads(sf.read_text()) if sf.is_file() else {}
    st[phase] = status
    sf.write_text(_json.dumps(st, indent=1))
def need(phase):
    sf = OUT / "state.json"
    st = _json.loads(sf.read_text()) if sf.is_file() else {}
    if st.get(phase) != "ok":
        raise RuntimeError(f"hard gate: phase '{phase}' is not ok (state={st.get(phase)}); refusing to continue")


In [ ]:
# BOOTSTRAP — offline, idempotent, always safe to run. Reconstructs POC +
# repo-dependency closure from embedded blobs (exact commit 981dd82 bytes).
import base64 as _b64, hashlib as _hl
BUNDLED = [{"path": "model.py", "root": "POC", "sha256": "2d0868e211f2896ff95010ad1b3993ee80e02f4691b21358dae6efe568d0e2e8", "b64": "IiIiVDUtc21hbGwgcG9zZS10by10ZXh0IHByb3RvdHlwZSBtb2RlbCAoUkVTRUFSQ0ggT05MWSwgbmV2ZXIgc2VydmluZykuCgpBcmNoaXRlY3R1cmU6CiAgICBbQiwgVCwgMTYyOV0gLT4gUG9zZVByb2plY3Rpb24gLT4gW0IsIFQsIDUxMl0gLT4gVDUgZW5jb2RlciAoaW5wdXRzX2VtYmVkcykKICAgIC0+IFQ1IGRlY29kZXIgLT4gRW5nbGlzaCB0b2tlbnMuCgpQb3NlIGlucHV0IGlzIHRoZSBTQU1FIG5vcm1hbGl6ZWQgW1QsIDE2MjldIHRoZSBwcm9kdWN0aW9uIGJhY2tlbmQgdXNlcwooYG5vcm1hbGl6ZV9wb3NlX3NlcXVlbmNlYCArIGByZXNpemVfc2VxdWVuY2VgIHRvIG1heCAxNjAgZnJhbWVzKS4gUGFkZGVkCmZyYW1lcyBhcmUgZXhjbHVkZWQgZnJvbSBlbmNvZGVyIGF0dGVudGlvbiB2aWEgYHBvc2VfbWFza2AuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbgoKUE9TRV9GRUFUVVJFX0RJTSA9IDE2MjkgICMgNTQzIGxhbmRtYXJrcyB4IDMsIGJhY2tlbmQgY2Fub25pY2FsClQ1X0RfTU9ERUwgPSA1MTIgICMgZ29vZ2xlLXQ1L3Q1LXNtYWxsIGRfbW9kZWwKVDVfQ0hFQ0tQT0lOVCA9ICJnb29nbGUtdDUvdDUtc21hbGwiCk1BWF9GUkFNRVMgPSAxNjAgICMgc2FtZSBhcyBiYWNrZW5kL1NpZ25TYWdlIHByZXByb2Nlc3NpbmcgY29udmVudGlvbgpNQVhfVE9LRU5TID0gOTYgICMgc2FtZSBhcyBleGlzdGluZyBWMiB0cmFpbmluZwoKCmNsYXNzIFBvc2VQcm9qZWN0aW9uKG5uLk1vZHVsZSk6CiAgICAiIiJMaWdodHdlaWdodCBwb3NlIGZyb250LWVuZDogTGluZWFyKDE2MjktPjUxMikgKyBMTiArIEdFTFUgKyBMaW5lYXIoNTEyLT41MTIpLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9kaW06IGludCA9IFBPU0VfRkVBVFVSRV9ESU0sIGRfbW9kZWw6IGludCA9IFQ1X0RfTU9ERUwsCiAgICAgICAgICAgICAgICAgZHJvcG91dDogZmxvYXQgPSAwLjEpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubmV0ID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGluX2RpbSwgZF9tb2RlbCksCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShkX21vZGVsKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KGRyb3BvdXQpLAogICAgICAgICAgICBubi5MaW5lYXIoZF9tb2RlbCwgZF9tb2RlbCksCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShkX21vZGVsKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgcG9zZXM6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBzZWxmLm5ldChwb3NlcykKCgpjbGFzcyBQb3NlVDUobm4uTW9kdWxlKToKICAgICIiIlBvc2VQcm9qZWN0aW9uICsgcHJldHJhaW5lZCBUNS1zbWFsbCwgd2lyZWQgdGhyb3VnaCBgaW5wdXRzX2VtYmVkc2AuCgogICAgUG9zZSB2YWx1ZXMgTkVWRVIgZW50ZXIgdGhlIFQ1IHRva2VuIGVtYmVkZGluZyBsYXllci4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjaGVja3BvaW50OiBzdHIgPSBUNV9DSEVDS1BPSU5ULCBkcm9wb3V0OiBmbG9hdCA9IDAuMSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5wcm9qZWN0aW9uID0gUG9zZVByb2plY3Rpb24oZHJvcG91dD1kcm9wb3V0KQogICAgICAgIHNlbGYudDUgPSBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbi5mcm9tX3ByZXRyYWluZWQoY2hlY2twb2ludCkKICAgICAgICBhc3NlcnQgc2VsZi50NS5jb25maWcuZF9tb2RlbCA9PSBUNV9EX01PREVMLCBzZWxmLnQ1LmNvbmZpZy5kX21vZGVsCgogICAgZGVmIGVuY29kZV9wb3NlcyhzZWxmLCBwb3NlczogdG9yY2guVGVuc29yLCBwb3NlX21hc2s6IHRvcmNoLlRlbnNvcik6CiAgICAgICAgIiIicG9zZXMgW0IsVCwxNjI5XSBmbG9hdDsgcG9zZV9tYXNrIFtCLFRdIGJvb2wsIFRydWUgPSByZWFsIGZyYW1lLiIiIgogICAgICAgIGVtYmVkcyA9IHNlbGYucHJvamVjdGlvbihwb3NlcykKICAgICAgICByZXR1cm4gc2VsZi50NS5lbmNvZGVyKGlucHV0X2lkcz1Ob25lLCBpbnB1dHNfZW1iZWRzPWVtYmVkcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPXBvc2VfbWFzay5mbG9hdCgpKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHBvc2VzOiB0b3JjaC5UZW5zb3IsIHBvc2VfbWFzazogdG9yY2guVGVuc29yLAogICAgICAgICAgICAgICAgbGFiZWxzOiB0b3JjaC5UZW5zb3IpOgogICAgICAgIGVtYmVkcyA9IHNlbGYucHJvamVjdGlvbihwb3NlcykKICAgICAgICByZXR1cm4gc2VsZi50NShpbnB1dHNfZW1iZWRzPWVtYmVkcywgYXR0ZW50aW9uX21hc2s9cG9zZV9tYXNrLmZsb2F0KCksCiAgICAgICAgICAgICAgICAgICAgICAgbGFiZWxzPWxhYmVscykKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgdHJhbnNsYXRlKHNlbGYsIHBvc2VzOiB0b3JjaC5UZW5zb3IsIHBvc2VfbWFzazogdG9yY2guVGVuc29yLAogICAgICAgICAgICAgICAgICB0b2tlbml6ZXIsIG1heF9uZXdfdG9rZW5zOiBpbnQgPSA5NCwgbnVtX2JlYW1zOiBpbnQgPSAxKSAtPiBsaXN0W3N0cl06CiAgICAgICAgc2VsZi5ldmFsKCkKICAgICAgICBlbWJlZHMgPSBzZWxmLnByb2plY3Rpb24ocG9zZXMpCiAgICAgICAgZW5jID0gc2VsZi50NS5lbmNvZGVyKGlucHV0X2lkcz1Ob25lLCBpbnB1dHNfZW1iZWRzPWVtYmVkcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9cG9zZV9tYXNrLmZsb2F0KCkpCiAgICAgICAgb3V0ID0gc2VsZi50NS5nZW5lcmF0ZShlbmNvZGVyX291dHB1dHM9ZW5jLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9cG9zZV9tYXNrLmZsb2F0KCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1tYXhfbmV3X3Rva2VucywgbnVtX2JlYW1zPW51bV9iZWFtcykKICAgICAgICByZXR1cm4gdG9rZW5pemVyLmJhdGNoX2RlY29kZShvdXQsIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkKCiAgICBkZWYgc2V0X3Q1X2Zyb3plbihzZWxmLCBmcm96ZW46IGJvb2wpIC0+IE5vbmU6CiAgICAgICAgZm9yIHAgaW4gc2VsZi50NS5wYXJhbWV0ZXJzKCk6CiAgICAgICAgICAgIHAucmVxdWlyZXNfZ3JhZCA9IGZyb3plbiBpcyBGYWxzZQoKICAgIGRlZiB0cmFpbmFibGVfc3VtbWFyeShzZWxmKSAtPiBkaWN0OgogICAgICAgIHRvdGFsID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKSkKICAgICAgICB0cmFpbmFibGUgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIHNlbGYucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkKICAgICAgICByZXR1cm4geyJ0b3RhbCI6IHRvdGFsLCAidHJhaW5hYmxlIjogdHJhaW5hYmxlfQo="}, {"path": "materialize.py", "root": "POC", "sha256": "4111cec3e1e118905bbc7a8258e9125852eef82b9c464928c6b2d20d4ee3918e", "b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJNYXRlcmlhbGl6ZSBhIFNNQUxMICh+NWspIGxvY2FsIHBvc2Ugc3Vic2V0IGZvciB0aGUgVDUgUE9DIChSRVNFQVJDSCBPTkxZKS4KClJldXNlcywgaW4gb3JkZXI6Ci0gYGNsZWFuX2RhdGFmcmFtZWAgKyBgdmlkZW9fc3BsaXRgIGZyb20gcmVzZWFyY2gvdHJhaW5pbmcvYXVkaXRfZGF0YXNldC5weQogIChvZmZpY2lhbCB2aWRlby1sZXZlbCBzcGxpdCBwb2xpY3k7IHN1YnNldCBrZWVwcyB3aG9sZSB2aWRlb3Mgb25seSkKLSBgcmVhZF9ub3JtYWxpemVkX3Bvc2VgICsgYHJlc2l6ZV9zZXF1ZW5jZWAgcGF0aCBmcm9tCiAgcmVzZWFyY2gvdHJhaW5pbmcvdHJhaW5fc2lnbl90b190ZXh0LnB5IChjYW5vbmljYWwgcG9zZV9ub3JtYWxpemVfdjEsCiAgTUFYX0ZSQU1FUz0xNjAsIFsxNjAsMTYyOV0gKyB0cnVlIGxlbmd0aCkKCkZvcm1hdDogb25lIC5ucHogc2hhcmQgcGVyIHNwbGl0IGNodW5rIChmYXN0LCBkZXRlcm1pbmlzdGljLCBpbnNwZWN0YWJsZSk6CiAgICBwb3NlcyAgIGZsb2F0MTYgW04sMTYwLDE2MjldICAgbm9ybWFsaXplZCArIHJlc2l6ZWQKICAgIGxlbmd0aHMgaW50MzIgICBbTl0gICAgICAgICAgICB0cnVlICh1bnBhZGRlZCkgbGVuZ3RocwogICAgdGV4dHMgICAnPFUuLi4nIFtOXSAgICAgICAgICAgIG9yaWdpbmFsIEVuZ2xpc2ggdHJhbnNsYXRpb25zCiAgICB1aWRzICAgICc8VS4uLicgW05dCiAgICB2aWRlb19pZHMgJzxVLi4uJyBbTl0KClJlc3VtYWJpbGl0eTogc2hhcmRzIGFyZSB3cml0dGVuIGF0b21pY2FsbHkgKHRtcCArIHJlbmFtZSkgYW5kIHRoZSBtYW5pZmVzdAppcyB1cGRhdGVkIGFmdGVyIEVWRVJZIHNoYXJkLiBBIHJlc3RhcnQgcmUtdmFsaWRhdGVzIHJlY29yZGVkIHNoYXJkcyB2aWEKc2hhMjU2LCBza2lwcyByb3dzIGFscmVhZHkgbWF0ZXJpYWxpemVkLCBhbmQgY29udGludWVzLiBSZWxhdGl2ZSBzaGFyZCBwYXRocwprZWVwIHRoZSBtYW5pZmVzdCBwb3J0YWJsZSBhY3Jvc3MgS2FnZ2xlIHJlLWF0dGFjaGVzLgoKVXNhZ2UgKEthZ2dsZSBHUFUgaG9zdCwgSEZfVE9LRU4gd2l0aCBpU2lnbiBhY2Nlc3MgaW4gZW52KToKICAgIHB5dGhvbiBtYXRlcmlhbGl6ZS5weSAtLWNzdiAvcGF0aC9pU2lnbl92MS4xLmNzdiAtLXBvc2Utc291cmNlIHJlbW90ZSBcXAogICAgICAgIC0tb3V0IC4vdDVfcG9jX3NoYXJkcyAtLXRhcmdldC10cmFpbiA0MDAwIC0tdGFyZ2V0LXZhbCA1MDAgLS10YXJnZXQtdGVzdCA1MDAKCkxvY2FsIHNtb2tlIHRlc3QgKG5vIGRhdGFzZXQsIG5vIHRva2VuOyBwcm92ZXMgd3JpdGUgKyBtYW5pZmVzdCArIHJlc3VtZSk6CiAgICBweXRob24gbWF0ZXJpYWxpemUucHkgLS1zbW9rZSAtLW91dCAvdG1wL3BvY19zbW9rZQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKClJFUE9fUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUkVQT19ST09UIC8gImJhY2tlbmQiKSkKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihSRVBPX1JPT1QgLyAicmVzZWFyY2giIC8gInRyYWluaW5nIikpCgpmcm9tIGF1ZGl0X2RhdGFzZXQgaW1wb3J0IGNsZWFuX2RhdGFmcmFtZSwgdmlkZW9fc3BsaXQgICMgbm9xYTogRTQwMgpmcm9tIHRyYWluX3NpZ25fdG9fdGV4dCBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBNQVhfRlJBTUVTLAogICAgX29wZW5fcG9zZV9zdG9yZSwKICAgIGxvYWRfcG9zZV9pdGVtLAogICAgcHJlZmxpZ2h0X3Bvc2Vfc291cmNlLAopCmZyb20gYXBwLm1sLnByZXByb2Nlc3Npbmcuc2VxdWVuY2UgaW1wb3J0IHJlc2l6ZV9zZXF1ZW5jZSAgIyBub3FhOiBFNDAyCgpQT1NFX0RJTSA9IDE2MjkKREVGQVVMVF9TSEFSRF9TSVpFID0gMTAwMApERUZBVUxUX01BWF9TS0lQX1JBVEUgPSAwLjA1CgoKZGVmIHNoYTI1Nl9maWxlKHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZmg6CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBmaC5yZWFkKDggKiAxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgYXRvbWljX3dyaXRlX25weihwYXRoOiBQYXRoLCAqKmFycmF5cykgLT4gTm9uZToKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgoIi50bXAubnB6IikKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQodG1wLCAqKmFycmF5cykKICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQoKCmRlZiBhdG9taWNfd3JpdGVfanNvbihwYXRoOiBQYXRoLCBvYmo6IGRpY3QpIC0+IE5vbmU6CiAgICB0bXAgPSBwYXRoLndpdGhfc3VmZml4KCIudG1wLmpzb24iKQogICAgdG1wLndyaXRlX3RleHQoanNvbi5kdW1wcyhvYmosIGluZGVudD0xKSkKICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQoKCmRlZiB0YWtlX3ZpZGVvX3N1YnNldChkZjogcGQuRGF0YUZyYW1lLCB0YXJnZXRfcm93czogaW50LCBzZWVkOiBpbnQpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIldob2xlIHZpZGVvcyBvbmx5LCBzaHVmZmxlZCB3aXRoIGZpeGVkIHNlZWQsIHVudGlsIH50YXJnZXRfcm93cy4iIiIKICAgIHZpZHMgPSBkZlsidmlkZW9faWQiXS5kcm9wbmEoKS51bmlxdWUoKS50b2xpc3QoKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBybmcuc2h1ZmZsZSh2aWRzKQogICAgcGlja2VkLCBjb3VudCA9IFtdLCAwCiAgICBmb3IgdiBpbiB2aWRzOgogICAgICAgIGlmIGNvdW50ID49IHRhcmdldF9yb3dzOgogICAgICAgICAgICBicmVhawogICAgICAgIHBpY2tlZC5hcHBlbmQodikKICAgICAgICBjb3VudCArPSBpbnQoKGRmWyJ2aWRlb19pZCJdID09IHYpLnN1bSgpKQogICAgcmV0dXJuIGRmW2RmWyJ2aWRlb19pZCJdLmlzaW4ocGlja2VkKV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCmRlZiBzeW50aGV0aWNfcG9zZSh1aWQ6IHN0ciwgc2VlZDogaW50ID0gNDIpIC0+IHR1cGxlW25wLm5kYXJyYXksIGludF06CiAgICAiIiJEZXRlcm1pbmlzdGljIHN5bnRoZXRpYyBwb3NlOiBleGVyY2lzZXMgcmVzaXplL3RydWVfbGVuIHdpdGhvdXQgaVNpZ24uIiIiCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoYWJzKGhhc2goKHVpZCwgc2VlZCkpKSAlICgyICoqIDMxKSkKICAgIGZyYW1lcyA9IDQwICsgKGFicyhoYXNoKHVpZCkpICUgMTIwKSAgIyB2YXJ5aW5nIGxlbmd0aHM6IDQwLi4xNTkKICAgIHJhdyA9IHJuZy5ub3JtYWwoc2l6ZT0oZnJhbWVzLCBQT1NFX0RJTSkpLmFzdHlwZShucC5mbG9hdDMyKQogICAgdHJ1ZV9sZW4gPSBtaW4oZnJhbWVzLCBNQVhfRlJBTUVTKQogICAgcmV0dXJuIHJlc2l6ZV9zZXF1ZW5jZShyYXcsIE1BWF9GUkFNRVMpLmFzdHlwZShucC5mbG9hdDMyKSwgdHJ1ZV9sZW4KCgpkZWYgZmV0Y2hfcmVhbCh1aWQ6IHN0ciwgc3RvcmUpIC0+IHR1cGxlW25wLm5kYXJyYXksIGludF06CiAgICBmcmFtZXMsIHRydWVfbGVuID0gbG9hZF9wb3NlX2l0ZW0odWlkLCBzdG9yZSkKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShmcmFtZXMpLmFsbCgpIG9yIHRydWVfbGVuIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImludmFsaWQgcG9zZSBmb3Ige3VpZH0iKQogICAgcmV0dXJuIGZyYW1lcywgdHJ1ZV9sZW4KCgpkZWYgbG9hZF9zcGxpdF9tYW5pZmVzdChvdXQ6IFBhdGgsIG5hbWU6IHN0cikgLT4gZGljdDoKICAgICIiIkxvYWQgcHJpb3IgbWFuaWZlc3Q7IGtlZXAgb25seSBsZWFkaW5nIHJ1biBvZiB2YWxpZCBzaGFyZHMuIiIiCiAgICBwYXRoID0gb3V0IC8gZiJ7bmFtZX1fbWFuaWZlc3QuanNvbiIKICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByZXR1cm4geyJzcGxpdCI6IG5hbWUsICJzaGFyZHMiOiBbXSwgImNvbXBsZXRlZF91aWRzIjogW10sCiAgICAgICAgICAgICAgICAic2tpcHBlZCI6IFtdLCAibiI6IDB9CiAgICB0cnk6CiAgICAgICAgbWFuID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKQogICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgT1NFcnJvcik6CiAgICAgICAgcmV0dXJuIHsic3BsaXQiOiBuYW1lLCAic2hhcmRzIjogW10sICJjb21wbGV0ZWRfdWlkcyI6IFtdLAogICAgICAgICAgICAgICAgInNraXBwZWQiOiBbXSwgIm4iOiAwfQogICAgdmFsaWRfc2hhcmRzLCB2YWxpZF91aWRzID0gW10sIFtdCiAgICBmb3Igc2ggaW4gbWFuLmdldCgic2hhcmRzIiwgW10pOgogICAgICAgIHAgPSBvdXQgLyBzaFsiZmlsZSJdCiAgICAgICAgaWYgbm90IHAuaXNfZmlsZSgpIG9yIHNoYTI1Nl9maWxlKHApICE9IHNoLmdldCgic2hhMjU2Iik6CiAgICAgICAgICAgIGJyZWFrICAjIHRydW5jYXRlIGF0IGZpcnN0IGludmFsaWQ7IHJlc3QgYXJlIHVudHJ1c3RlZAogICAgICAgIHZhbGlkX3NoYXJkcy5hcHBlbmQoc2gpCiAgICAgICAgdmFsaWRfdWlkcy5leHRlbmQoc2guZ2V0KCJ1aWRzIiwgW10pKQogICAgbWFuWyJzaGFyZHMiXSA9IHZhbGlkX3NoYXJkcwogICAgbWFuWyJjb21wbGV0ZWRfdWlkcyJdID0gdmFsaWRfdWlkcwogICAgbWFuWyJuIl0gPSBzdW0oc1sibiJdIGZvciBzIGluIHZhbGlkX3NoYXJkcykKICAgIHJldHVybiBtYW4KCgpkZWYgZmx1c2hfc2hhcmQob3V0OiBQYXRoLCBuYW1lOiBzdHIsIGlkeDogaW50LCBidWY6IGRpY3QsIG1hbjogZGljdCkgLT4gTm9uZToKICAgIHBvc2VzID0gbnAuc3RhY2soYnVmWyJwb3NlcyJdKS5hc3R5cGUobnAuZmxvYXQxNikKICAgIHBhdGggPSBvdXQgLyBmIntuYW1lfV97aWR4OjAzZH0ubnB6IgogICAgYXRvbWljX3dyaXRlX25weigKICAgICAgICBwYXRoLAogICAgICAgIHBvc2VzPXBvc2VzLAogICAgICAgIGxlbmd0aHM9bnAuYXNhcnJheShidWZbImxlbmd0aHMiXSwgZHR5cGU9bnAuaW50MzIpLAogICAgICAgIHRleHRzPW5wLmFzYXJyYXkoYnVmWyJ0ZXh0cyJdKSwKICAgICAgICB1aWRzPW5wLmFzYXJyYXkoYnVmWyJ1aWRzIl0pLAogICAgICAgIHZpZGVvX2lkcz1ucC5hc2FycmF5KGJ1ZlsidmlkZW9faWRzIl0pLAogICAgKQogICAgbWFuWyJzaGFyZHMiXS5hcHBlbmQoewogICAgICAgICJmaWxlIjogcGF0aC5uYW1lLCAgIyByZWxhdGl2ZTogcG9ydGFibGUgYWNyb3NzIEthZ2dsZSByZS1hdHRhY2hlcwogICAgICAgICJuIjogbGVuKGJ1ZlsidWlkcyJdKSwKICAgICAgICAic2hhMjU2Ijogc2hhMjU2X2ZpbGUocGF0aCksCiAgICAgICAgImR0eXBlIjogImZsb2F0MTYiLAogICAgICAgICJzaGFwZSI6IFtpbnQoeCkgZm9yIHggaW4gcG9zZXMuc2hhcGVdLAogICAgICAgICJ1aWRfZmlyc3QiOiBidWZbInVpZHMiXVswXSwKICAgICAgICAidWlkX2xhc3QiOiBidWZbInVpZHMiXVstMV0sCiAgICAgICAgInVpZHMiOiBsaXN0KGJ1ZlsidWlkcyJdKSwKICAgIH0pCiAgICBtYW5bImNvbXBsZXRlZF91aWRzIl0uZXh0ZW5kKGJ1ZlsidWlkcyJdKQogICAgbWFuWyJuIl0gPSBzdW0oc1sibiJdIGZvciBzIGluIG1hblsic2hhcmRzIl0pCiAgICBhdG9taWNfd3JpdGVfanNvbihvdXQgLyBmIntuYW1lfV9tYW5pZmVzdC5qc29uIiwgbWFuKQogICAgcHJpbnQoZiJbe25hbWV9XSBzaGFyZCB7cGF0aC5uYW1lfTogbj17bGVuKGJ1ZlsndWlkcyddKX0iLCBmbHVzaD1UcnVlKQoKCmRlZiBtYXRlcmlhbGl6ZV9zcGxpdChkZjogcGQuRGF0YUZyYW1lLCBmZXRjaCwgb3V0OiBQYXRoLCBuYW1lOiBzdHIsCiAgICAgICAgICAgICAgICAgICAgICBzZWVkOiBpbnQsIHNoYXJkX3NpemU6IGludCwKICAgICAgICAgICAgICAgICAgICAgIG1heF9za2lwX3JhdGU6IGZsb2F0KSAtPiBkaWN0OgogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG1hbiA9IGxvYWRfc3BsaXRfbWFuaWZlc3Qob3V0LCBuYW1lKQogICAgZG9uZSA9IHNldChtYW5bImNvbXBsZXRlZF91aWRzIl0pCiAgICBpZiBkb25lOgogICAgICAgIHByaW50KGYiW3tuYW1lfV0gcmVzdW1lOiB7bGVuKGRvbmUpfSByb3dzIGFscmVhZHkgbWF0ZXJpYWxpemVkLCAiCiAgICAgICAgICAgICAgZiJ7bGVuKG1hblsnc2hhcmRzJ10pfSB2YWxpZCBzaGFyZHMiLCBmbHVzaD1UcnVlKQogICAgc2tpcHBlZCA9IG1hbi5nZXQoInNraXBwZWQiLCBbXSkKICAgIGJ1ZjogZGljdCA9IHtrOiBbXSBmb3IgayBpbiAoInBvc2VzIiwgImxlbmd0aHMiLCAidGV4dHMiLCAidWlkcyIsICJ2aWRlb19pZHMiKX0KICAgIGlkeCA9IGxlbihtYW5bInNoYXJkcyJdKQogICAgcHJvY2Vzc2VkID0gMAogICAgZm9yIF8sIHJvdyBpbiBkZi5pdGVycm93cygpOgogICAgICAgIHVpZCA9IHN0cihyb3dbInVpZCJdKQogICAgICAgIGlmIHVpZCBpbiBkb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRleHQgPSBzdHIocm93WyJ0ZXh0Il0pCiAgICAgICAgcHJvY2Vzc2VkICs9IDEKICAgICAgICBpZiBub3QgdGV4dC5zdHJpcCgpOgogICAgICAgICAgICBza2lwcGVkLmFwcGVuZCh7InVpZCI6IHVpZCwgInJlYXNvbiI6ICJlbXB0eV90ZXh0In0pCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcmFtZXMsIHRydWVfbGVuID0gZmV0Y2godWlkKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEgLSByZWNvcmRlZCwgbmV2ZXIgc2lsZW50CiAgICAgICAgICAgIHNraXBwZWQuYXBwZW5kKHsidWlkIjogdWlkLCAicmVhc29uIjogZiJmZXRjaF9mYWlsZWQ6IHtleGN9In0pCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYnVmWyJwb3NlcyJdLmFwcGVuZChmcmFtZXMuYXN0eXBlKG5wLmZsb2F0MTYpKQogICAgICAgIGJ1ZlsibGVuZ3RocyJdLmFwcGVuZCh0cnVlX2xlbikKICAgICAgICBidWZbInRleHRzIl0uYXBwZW5kKHRleHQpCiAgICAgICAgYnVmWyJ1aWRzIl0uYXBwZW5kKHVpZCkKICAgICAgICBidWZbInZpZGVvX2lkcyJdLmFwcGVuZChzdHIocm93WyJ2aWRlb19pZCJdKSkKICAgICAgICBpZiBsZW4oYnVmWyJ1aWRzIl0pID49IHNoYXJkX3NpemU6CiAgICAgICAgICAgIGZsdXNoX3NoYXJkKG91dCwgbmFtZSwgaWR4LCBidWYsIG1hbikKICAgICAgICAgICAgaWR4ICs9IDEKICAgICAgICAgICAgYnVmID0ge2s6IFtdIGZvciBrIGluIGJ1Zn0KICAgIGlmIGJ1ZlsidWlkcyJdOgogICAgICAgIGZsdXNoX3NoYXJkKG91dCwgbmFtZSwgaWR4LCBidWYsIG1hbikKICAgICAgICBpZHggKz0gMQogICAgbWFuWyJza2lwcGVkIl0gPSBza2lwcGVkCiAgICBtYW5bInNlZWQiXSA9IHNlZWQKICAgIG1hblsic2hhcmRfc2l6ZSJdID0gc2hhcmRfc2l6ZQogICAgdG90YWwgPSBtYW5bIm4iXSArIGxlbihza2lwcGVkKQogICAgbWFuWyJza2lwX3JhdGUiXSA9IGxlbihza2lwcGVkKSAvIHRvdGFsIGlmIHRvdGFsIGVsc2UgMS4wCiAgICBhdG9taWNfd3JpdGVfanNvbihvdXQgLyBmIntuYW1lfV9tYW5pZmVzdC5qc29uIiwgbWFuKQogICAgcHJpbnQoZiJbe25hbWV9XSBrZXB0PXttYW5bJ24nXX0gc2tpcHBlZD17bGVuKHNraXBwZWQpfSAiCiAgICAgICAgICBmInNraXBfcmF0ZT17bWFuWydza2lwX3JhdGUnXTouMyV9IHNoYXJkcz17bGVuKG1hblsnc2hhcmRzJ10pfSIsIGZsdXNoPVRydWUpCiAgICBpZiBtYW5bIm4iXSA9PSAwOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJbe25hbWV9XSBGQUlMOiB6ZXJvIHNhbXBsZXMgbWF0ZXJpYWxpemVkLiIpCiAgICBpZiBtYW5bInNraXBfcmF0ZSJdID4gbWF4X3NraXBfcmF0ZToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYiW3tuYW1lfV0gRkFJTDogc2tpcF9yYXRlIHttYW5bJ3NraXBfcmF0ZSddOi4zJX0gPiAiCiAgICAgICAgICAgICAgICAgICAgICAgICBmInttYXhfc2tpcF9yYXRlOi4zJX0gLS0gcmVmdXNpbmcgdG8gY2xhaW0gc3VjY2Vzcy4iKQogICAgcmV0dXJuIG1hbgoKCmRlZiBzYW5pdHlfcHJpbnQoc2hhcmRfcGF0aDogc3RyLCBrOiBpbnQgPSAxMCwgc2VlZDogaW50ID0gNDIpIC0+IE5vbmU6CiAgICAiIiLCpzggZGF0YSBzYW5pdHkgY2hlY2s6IHNoYXBlcywgc3RhdHMsIGZpbml0ZW5lc3MsIG5vbi1lbXB0eSB0ZXh0LiIiIgogICAgeiA9IG5wLmxvYWQoc2hhcmRfcGF0aCwgYWxsb3dfcGlja2xlPVRydWUpCiAgICBuID0gbGVuKHpbInVpZHMiXSkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgZm9yIGkgaW4gcm5nLmNob2ljZShuLCBzaXplPW1pbihrLCBuKSwgcmVwbGFjZT1GYWxzZSk6CiAgICAgICAgcCA9IHpbInBvc2VzIl1baV0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgcHJpbnQoZiJ1aWQ9e3pbJ3VpZHMnXVtpXX0gdmlkZW89e3pbJ3ZpZGVvX2lkcyddW2ldfSBzaGFwZT17cC5zaGFwZX0gIgogICAgICAgICAgICAgIGYibGVuPXt6WydsZW5ndGhzJ11baV19IG1pbj17cC5taW4oKTouM2Z9IG1heD17cC5tYXgoKTouM2Z9ICIKICAgICAgICAgICAgICBmIm1lYW49e3AubWVhbigpOi40Zn0gc3RkPXtwLnN0ZCgpOi40Zn0gIgogICAgICAgICAgICAgIGYibmFuaW5mPXtib29sKCh+bnAuaXNmaW5pdGUocCkpLmFueSgpKX0gdGV4dD17elsndGV4dHMnXVtpXSFyfSIpCiAgICAgICAgYXNzZXJ0IHAuc2hhcGUgPT0gKE1BWF9GUkFNRVMsIFBPU0VfRElNKSBhbmQgelsibGVuZ3RocyJdW2ldIDw9IE1BWF9GUkFNRVMKICAgICAgICBhc3NlcnQgbnAuaXNmaW5pdGUocCkuYWxsKCkgYW5kIGJvb2woc3RyKHpbInRleHRzIl1baV0pLnN0cmlwKCkpCgoKZGVmIHNtb2tlX2RhdGFmcmFtZShuX3ZpZGVvczogaW50ID0gMzAsIHBlcl92aWRlbzogaW50ID0gMikgLT4gcGQuRGF0YUZyYW1lOgogICAgcm93cyA9IFtdCiAgICBmb3IgdiBpbiByYW5nZShuX3ZpZGVvcyk6CiAgICAgICAgZm9yIHMgaW4gcmFuZ2UocGVyX3ZpZGVvKToKICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJ1aWQiOiBmInNtb2tlLXZ7djowM2R9LXtzfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAidGV4dCI6IGYiU21va2UgdGVzdCBzZW50ZW5jZSB7c30gZm9yIHZpZGVvIHt2fSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAidmlkZW9faWQiOiBmInNtb2tlLXZ7djowM2R9In0pCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpCgoKZGVmIG1haW4oYXJndj1Ob25lKSAtPiBpbnQ6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJNYXRlcmlhbGl6ZSBzbWFsbCBUNSBQT0Mgc3Vic2V0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jc3YiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcG9zZS1zb3VyY2UiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ibG9jYWwgLnBvc2UgZGlyLCBsb2NhbCAuemlwLCBvciAncmVtb3RlJyAobmVlZHMgSEZfVE9LRU4pIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhcmdldC10cmFpbiIsIHR5cGU9aW50LCBkZWZhdWx0PTQwMDApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGFyZ2V0LXZhbCIsIHR5cGU9aW50LCBkZWZhdWx0PTUwMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YXJnZXQtdGVzdCIsIHR5cGU9aW50LCBkZWZhdWx0PTUwMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1saW1pdCIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iY2FwIHJvd3MgcGVyIHNwbGl0IChxdWljayByZWFsLXNvdXJjZSB0ZXN0KSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2hhcmQtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PURFRkFVTFRfU0hBUkRfU0laRSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXgtc2tpcC1yYXRlIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1ERUZBVUxUX01BWF9TS0lQX1JBVEUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNhbml0eSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc21va2UiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InN5bnRoZXRpYyBsb2NhbCBydW46IG5vIENTViwgbm8gcG9zZXMsIG5vIHRva2VuIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKGFyZ3YpCgogICAgb3V0ID0gUGF0aChhcmdzLm91dCkKCiAgICBpZiBhcmdzLnNtb2tlOgogICAgICAgIHByaW50KCJTTU9LRSBtb2RlOiBzeW50aGV0aWMgcG9zZXMsIGV4ZXJjaXNlcyBzaGFyZGluZyArIG1hbmlmZXN0ICsgcmVzdW1lLiIsIGZsdXNoPVRydWUpCiAgICAgICAgY2xlYW5lZCA9IHNtb2tlX2RhdGFmcmFtZSgpCiAgICAgICAgY2xlYW5lZFsidmlkZW9faWQiXSA9IGNsZWFuZWRbInZpZGVvX2lkIl0uYXN0eXBlKHN0cikKICAgICAgICBkZl90ciwgZGZfdmEsIGRmX3RlLCBzcGxpdCA9IHZpZGVvX3NwbGl0KGNsZWFuZWQsIGFyZ3Muc2VlZCkKICAgICAgICBmZXRjaCA9IGxhbWJkYSB1aWQ6IHN5bnRoZXRpY19wb3NlKHVpZCwgYXJncy5zZWVkKSAgIyBub3FhOiBFNzMxCiAgICBlbHNlOgogICAgICAgIGlmIG5vdCBhcmdzLmNzdiBvciBub3QgYXJncy5wb3NlX3NvdXJjZToKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1jc3YgYW5kIC0tcG9zZS1zb3VyY2UgYXJlIHJlcXVpcmVkIChvciB1c2UgLS1zbW9rZSkiKQogICAgICAgIGRmID0gcGQucmVhZF9jc3YoYXJncy5jc3YpCiAgICAgICAgY2xlYW5lZCwgY2xlYW5pbmcgPSBjbGVhbl9kYXRhZnJhbWUoZGYpCiAgICAgICAgaWYgY2xlYW5pbmdbImR1cGxpY2F0ZV91aWQiXToKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiZHVwbGljYXRlIFVJRHMgYWZ0ZXIgY2xlYW5pbmcgLS0gcmVmdXNpbmcuIikKICAgICAgICBkZl90ciwgZGZfdmEsIGRmX3RlLCBzcGxpdCA9IHZpZGVvX3NwbGl0KGNsZWFuZWQsIGFyZ3Muc2VlZCkKICAgICAgICBwcmludChmIm9mZmljaWFsIHZpZGVvLWxldmVsIHNwbGl0OiB0cmFpbj17bGVuKGRmX3RyKX0gdmFsPXtsZW4oZGZfdmEpfSAiCiAgICAgICAgICAgICAgZiJ0ZXN0PXtsZW4oZGZfdGUpfSB2aWRlb3M9e3NwbGl0WyduX3ZpZGVvcyddfSIsIGZsdXNoPVRydWUpCiAgICAgICAga2luZCA9ICJyZW1vdGUiIGlmIGFyZ3MucG9zZV9zb3VyY2UgPT0gInJlbW90ZSIgZWxzZSAoCiAgICAgICAgICAgICJ6aXAiIGlmIGFyZ3MucG9zZV9zb3VyY2UuZW5kc3dpdGgoIi56aXAiKSBlbHNlICJkaXIiKQogICAgICAgIGFyZyA9IG9zLmVudmlyb24uZ2V0KCJIRl9UT0tFTiIsICIiKSBpZiBraW5kID09ICJyZW1vdGUiIGVsc2UgYXJncy5wb3NlX3NvdXJjZQogICAgICAgIGlmIGtpbmQgPT0gInJlbW90ZSIgYW5kIG5vdCBhcmc6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoInJlbW90ZSBwb3NlIHN0cmVhbWluZyBuZWVkcyBIRl9UT0tFTiIpCiAgICAgICAgc3RvcmUgPSBfb3Blbl9wb3NlX3N0b3JlKGtpbmQsIGFyZykKICAgICAgICBwcmVmbGlnaHRfcG9zZV9zb3VyY2Uoc3RvcmUsIHN0cihkZl90clsidWlkIl0uaWxvY1swXSkpCiAgICAgICAgZmV0Y2ggPSBsYW1iZGEgdWlkOiBmZXRjaF9yZWFsKHVpZCwgc3RvcmUpICAjIG5vcWE6IEU3MzEKCiAgICB0YXJnZXRzID0geyJ0cmFpbiI6IGFyZ3MudGFyZ2V0X3RyYWluLCAidmFsIjogYXJncy50YXJnZXRfdmFsLCAidGVzdCI6IGFyZ3MudGFyZ2V0X3Rlc3R9CiAgICBmcmFtZXMgPSB7InRyYWluIjogZGZfdHIsICJ2YWwiOiBkZl92YSwgInRlc3QiOiBkZl90ZX0KICAgIHNlZWRzID0geyJ0cmFpbiI6IGFyZ3Muc2VlZCwgInZhbCI6IGFyZ3Muc2VlZCArIDEsICJ0ZXN0IjogYXJncy5zZWVkICsgMn0KICAgIHJlcG9ydDogZGljdCA9IHsibW9kZSI6ICJzbW9rZSIgaWYgYXJncy5zbW9rZSBlbHNlICJyZWFsIiwKICAgICAgICAgICAgICAgICAgICAic2VlZCI6IGFyZ3Muc2VlZCwgInNoYXJkX3NpemUiOiBhcmdzLnNoYXJkX3NpemUsCiAgICAgICAgICAgICAgICAgICAgInN1YnNldCI6IHt9fQogICAgaWYgbm90IGFyZ3Muc21va2U6CiAgICAgICAgcmVwb3J0WyJjbGVhbmluZyJdID0gY2xlYW5pbmcKICAgICAgICByZXBvcnRbInNwbGl0Il0gPSBzcGxpdAogICAgZm9yIG5hbWUgaW4gKCJ0cmFpbiIsICJ2YWwiLCAidGVzdCIpOgogICAgICAgIHN1YiA9IHRha2VfdmlkZW9fc3Vic2V0KGZyYW1lc1tuYW1lXSwgdGFyZ2V0c1tuYW1lXSwgc2VlZHNbbmFtZV0pCiAgICAgICAgaWYgYXJncy5saW1pdDoKICAgICAgICAgICAgc3ViID0gc3ViLmhlYWQoYXJncy5saW1pdCkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgICAgIHJlcG9ydFsic3Vic2V0Il1bbmFtZV0gPSB7CiAgICAgICAgICAgIGs6IHYgZm9yIGssIHYgaW4gbWF0ZXJpYWxpemVfc3BsaXQoCiAgICAgICAgICAgICAgICBzdWIsIGZldGNoLCBvdXQsIG5hbWUsIHNlZWRzW25hbWVdLAogICAgICAgICAgICAgICAgYXJncy5zaGFyZF9zaXplLCBhcmdzLm1heF9za2lwX3JhdGUpLml0ZW1zKCkKICAgICAgICAgICAgaWYgayAhPSAiY29tcGxldGVkX3VpZHMifQogICAgYXRvbWljX3dyaXRlX2pzb24ob3V0IC8gIm1hdGVyaWFsaXplX3JlcG9ydC5qc29uIiwgcmVwb3J0KQogICAgaWYgYXJncy5zYW5pdHk6CiAgICAgICAgZm9yIG5hbWUgaW4gKCJ0cmFpbiIsICJ2YWwiLCAidGVzdCIpOgogICAgICAgICAgICBmb3Igc2hhcmQgaW4gcmVwb3J0WyJzdWJzZXQiXVtuYW1lXVsic2hhcmRzIl1bOjFdOgogICAgICAgICAgICAgICAgc2FuaXR5X3ByaW50KHN0cihvdXQgLyBzaGFyZFsiZmlsZSJdKSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK"}, {"path": "train.py", "root": "POC", "sha256": "80bb4b528c8a44b9c58d17b6219bd817ff1caa6b5cb40d74e9cdafc26e32e242", "b64": "..."}, {"path": "evaluate.py", "root": "POC", "sha256": "bcc3f906819f447f0dd34ff60fc3a3cf4cd8b2be0f65c35aa11579a63a1f906d", "b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJFdmFsdWF0ZSBhIHRyYWluZWQgUG9zZVQ1IGNoZWNrcG9pbnQgb24gaGVsZC1vdXQgc2hhcmRzIChSRVNFQVJDSCBPTkxZKS4KClJlcG9ydHM6IFdFUiwgQkxFVS0xLi40LCBST1VHRS1MLCBleGFjdCBtYXRjaCwgbm9uLWVtcHR5IHJhdGUsIHByZWRpY3Rpb24KZGl2ZXJzaXR5LCBtZWFuIGxlbmd0aHMsIHRyaXZpYWwgYmFzZWxpbmUsIDMwIHF1YWxpdGF0aXZlIGV4YW1wbGVzLApsYW5ndWFnZS1wcmlvci9jb2xsYXBzZSBzdGF0cywgYW5kIGlucHV0LXNlbnNpdGl2aXR5IHRlc3RzCihwb3NlIHN3YXAgLyB6ZXJvIGlucHV0IC8gdGVtcG9yYWwgc2h1ZmZsZSAtPiBQT1NFLUNPTkRJVElPTklORyBjaGVjaykuCgpVc2FnZToKICAgIHB5dGhvbiBldmFsdWF0ZS5weSAtLWNoZWNrcG9pbnQgLi90NV9wb2NfNWsvdDVfcG9zZV9wcm9qZWN0aW9uLnB0IFxcCiAgICAgICAgLS1zaGFyZHMgLi90NV9wb2Nfc2hhcmRzIC0tb3V0IC4vdDVfcG9jX2V2YWwKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc3lzCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgVDVUb2tlbml6ZXIKCmZyb20gbW9kZWwgaW1wb3J0IFQ1X0NIRUNLUE9JTlQsIFBvc2VUNQpmcm9tIHRyYWluIGltcG9ydCBTaGFyZERhdGFzZXQsIGNvbGxhdGVfcG9zZV90ZXh0LCBwb3NlX21hc2sKCmZyb20gZnVuY3Rvb2xzIGltcG9ydCBwYXJ0aWFsCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgZGVjb2RlX2FsbChtb2RlbDogUG9zZVQ1LCBsb2FkZXIsIHRva2VuaXplciwgZGV2aWNlKSAtPiB0dXBsZVtsaXN0LCBsaXN0LCBsaXN0XToKICAgIHJlZnMsIGh5cHMsIHVpZHMgPSBbXSwgW10sIFtdCiAgICBtb2RlbC5ldmFsKCkKICAgIGZvciBwb3NlcywgbGVuZ3RocywgbGFiZWxzLCBiYXRjaF91aWRzIGluIGxvYWRlcjoKICAgICAgICBwb3NlcyA9IHBvc2VzLnRvKGRldmljZSkKICAgICAgICBtYXNrID0gcG9zZV9tYXNrKGxlbmd0aHMsIHBvc2VzLnNpemUoMSksIGRldmljZSkKICAgICAgICBwcmVkcyA9IG1vZGVsLnRyYW5zbGF0ZShwb3NlcywgbWFzaywgdG9rZW5pemVyKQogICAgICAgIHJlZl9pZHMgPSBsYWJlbHMuY2xvbmUoKQogICAgICAgIHJlZl9pZHNbcmVmX2lkcyA9PSAtMTAwXSA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgICAgICByZWZzICs9IHRva2VuaXplci5iYXRjaF9kZWNvZGUocmVmX2lkcywgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgICAgIGh5cHMgKz0gcHJlZHMKICAgICAgICB1aWRzICs9IGJhdGNoX3VpZHMKICAgIHJldHVybiByZWZzLCBoeXBzLCB1aWRzCgoKZGVmIG5vcm0oczogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gIiAiLmpvaW4oc3RyKHMpLmxvd2VyKCkuc3RyaXAoKS5zcGxpdCgpKQoKCmRlZiBjb21wdXRlX21ldHJpY3MocmVmczogbGlzdFtzdHJdLCBoeXBzOiBsaXN0W3N0cl0pIC0+IGRpY3Q6CiAgICBpbXBvcnQgaml3ZXIKICAgIGZyb20gbmx0ay50cmFuc2xhdGUuYmxldV9zY29yZSBpbXBvcnQgY29ycHVzX2JsZXUsIFNtb290aGluZ0Z1bmN0aW9uCiAgICBmcm9tIHJvdWdlX3Njb3JlIGltcG9ydCByb3VnZV9zY29yZXIKCiAgICByID0gW25vcm0oeCkuc3BsaXQoKSBmb3IgeCBpbiByZWZzXQogICAgaCA9IFtub3JtKHgpLnNwbGl0KCkgZm9yIHggaW4gaHlwc10KICAgIHNtb290aCA9IFNtb290aGluZ0Z1bmN0aW9uKCkubWV0aG9kMQogICAgYmxldSA9IHt9CiAgICBmb3IgbiBpbiAoMSwgMiwgMywgNCk6CiAgICAgICAgdyA9IHR1cGxlKFsxIC8gbl0gKiBuICsgWzBdICogKDQgLSBuKSkKICAgICAgICBibGV1W2YiQkxFVS17bn0iXSA9IGZsb2F0KGNvcnB1c19ibGV1KAogICAgICAgICAgICBbW3hdIGZvciB4IGluIHJdLCBoLCB3ZWlnaHRzPXcsIHNtb290aGluZ19mdW5jdGlvbj1zbW9vdGgpKQogICAgc2NvcmVyID0gcm91Z2Vfc2NvcmVyLlJvdWdlU2NvcmVyKFsicm91Z2VMIl0sIHVzZV9zdGVtbWVyPVRydWUpCiAgICByb3VnZV9sID0gZmxvYXQobnAubWVhbihbc2NvcmVyLnNjb3JlKGEsIGIpWyJyb3VnZUwiXS5mbWVhc3VyZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBhLCBiIGluIHppcChyZWZzLCBoeXBzKV0pKQogICAgcmV0dXJuIHsKICAgICAgICAiV0VSIjogZmxvYXQoaml3ZXIud2VyKFtub3JtKHgpIGZvciB4IGluIHJlZnNdLCBbbm9ybSh4KSBmb3IgeCBpbiBoeXBzXSkpLAogICAgICAgICJST1VHRS1MIjogcm91Z2VfbCwKICAgICAgICAiZXhhY3RfbWF0Y2giOiBmbG9hdChucC5tZWFuKFtub3JtKGEpID09IG5vcm0oYikgZm9yIGEsIGIgaW4gemlwKHJlZnMsIGh5cHMpXSkpLAogICAgICAgICJub25fZW1wdHlfcmF0ZSI6IGZsb2F0KG5wLm1lYW4oW2Jvb2wobm9ybShiKSkgZm9yIGIgaW4gaHlwc10pKSwKICAgICAgICAiZ2VuX2xlbl9tZWFuIjogZmxvYXQobnAubWVhbihbbGVuKG5vcm0oYikuc3BsaXQoKSkgZm9yIGIgaW4gaHlwc10pKSwKICAgICAgICAicmVmX2xlbl9tZWFuIjogZmxvYXQobnAubWVhbihbbGVuKG5vcm0oYSkuc3BsaXQoKSkgZm9yIGEgaW4gcmVmc10pKSwKICAgICAgICAqKmJsZXUsCiAgICB9CgoKZGVmIGRpdmVyc2l0eShoeXBzOiBsaXN0W3N0cl0pIC0+IGRpY3Q6CiAgICB0b3AsIGNvdW50ID0gQ291bnRlcihoeXBzKS5tb3N0X2NvbW1vbigxKVswXSBpZiBoeXBzIGVsc2UgKCIiLCAwKQogICAgcmV0dXJuIHsidW5pcXVlIjogbGVuKHNldChoeXBzKSksCiAgICAgICAgICAgICJ1bmlxdWVfZnJhY3Rpb24iOiBsZW4oc2V0KGh5cHMpKSAvIG1heCgxLCBsZW4oaHlwcykpLAogICAgICAgICAgICAidG9wX3ByZWRpY3Rpb24iOiB0b3AsCiAgICAgICAgICAgICJ0b3BfcHJlZGljdGlvbl9mcmVxIjogY291bnQgLyBtYXgoMSwgbGVuKGh5cHMpKX0KCgpkZWYgc2Vuc2l0aXZpdHkobW9kZWw6IFBvc2VUNSwgbG9hZGVyLCB0b2tlbml6ZXIsIGRldmljZSwgazogaW50ID0gNCkgLT4gZGljdDoKICAgICIiIlN3YXAgLyB6ZXJvIC8gc2h1ZmZsZSBwb3Nlczsgb3V0cHV0cyBTSE9VTEQgY2hhbmdlIGlmIHBvc2UtY29uZGl0aW9uZWQuIiIiCiAgICBtb2RlbC5ldmFsKCkKICAgIGJhdGNoID0gbmV4dChpdGVyKGxvYWRlcikpCiAgICBwb3NlcywgbGVuZ3RocywgXywgXyA9IGJhdGNoCiAgICBwb3NlcyA9IHBvc2VzWzprXS50byhkZXZpY2UpCiAgICBtYXNrID0gcG9zZV9tYXNrKGxlbmd0aHNbOmtdLCBwb3Nlcy5zaXplKDEpLCBkZXZpY2UpCiAgICBiYXNlID0gbW9kZWwudHJhbnNsYXRlKHBvc2VzLCBtYXNrLCB0b2tlbml6ZXIpCiAgICBzd2FwcGVkID0gbW9kZWwudHJhbnNsYXRlKHBvc2VzLmZsaXAoMCksIG1hc2ssIHRva2VuaXplcikKICAgIHplcm9lZCA9IG1vZGVsLnRyYW5zbGF0ZSh0b3JjaC56ZXJvc19saWtlKHBvc2VzKSwgbWFzaywgdG9rZW5pemVyKQogICAgc2h1ZmZsZWQgPSBtb2RlbC50cmFuc2xhdGUoCiAgICAgICAgcG9zZXNbOiwgdG9yY2gucmFuZHBlcm0ocG9zZXMuc2l6ZSgxKSwgZGV2aWNlPXBvc2VzLmRldmljZSldLCBtYXNrLCB0b2tlbml6ZXIpCiAgICBzYW1lID0gbGFtYmRhIGEsIGI6IGZsb2F0KG5wLm1lYW4oW3guc3RyaXAoKSA9PSB5LnN0cmlwKCkgZm9yIHgsIHkgaW4gemlwKGEsIGIpXSkpICAjIG5vcWE6IEU3MzEKICAgIHJldHVybiB7ImJhc2UiOiBiYXNlLCAic3dhcHBlZCI6IHN3YXBwZWQsICJ6ZXJvZWQiOiB6ZXJvZWQsICJzaHVmZmxlZCI6IHNodWZmbGVkLAogICAgICAgICAgICAiZnJhY19zYW1lX3N3YXAiOiBzYW1lKGJhc2UsIHN3YXBwZWQpLAogICAgICAgICAgICAiZnJhY19zYW1lX3plcm8iOiBzYW1lKGJhc2UsIHplcm9lZCksCiAgICAgICAgICAgICJmcmFjX3NhbWVfc2h1ZmZsZSI6IHNhbWUoYmFzZSwgc2h1ZmZsZWQpLAogICAgICAgICAgICAicG9zZV9jb25kaXRpb25lZCI6IGJvb2woc2FtZShiYXNlLCBzd2FwcGVkKSA8IDEuMAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3Igc2FtZShiYXNlLCB6ZXJvZWQpIDwgMS4wCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBzYW1lKGJhc2UsIHNodWZmbGVkKSA8IDEuMCl9CgoKZGVmIG1haW4oYXJndj1Ob25lKSAtPiBpbnQ6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJFdmFsdWF0ZSBQb3NlVDUgY2hlY2twb2ludCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2hhcmRzIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1leGFtcGxlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTMwKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoYXJndikKCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG91dCA9IFBhdGgoYXJncy5vdXQpCiAgICBvdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdG9rZW5pemVyID0gVDVUb2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKFQ1X0NIRUNLUE9JTlQpCiAgICBja3B0ID0gdG9yY2gubG9hZChhcmdzLmNoZWNrcG9pbnQsIG1hcF9sb2NhdGlvbj1kZXZpY2UpCiAgICBtb2RlbCA9IFBvc2VUNSgpLnRvKGRldmljZSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChja3B0WyJzdGF0ZV9kaWN0Il0pCgogICAgc2hhcmRfZGlyID0gUGF0aChhcmdzLnNoYXJkcykKICAgIHRlc3RfZHMgPSBTaGFyZERhdGFzZXQoc29ydGVkKG1hcChzdHIsIHNoYXJkX2Rpci5nbG9iKCJ0ZXN0XyoubnB6IikpKSkKICAgIGxvYWRlciA9IERhdGFMb2FkZXIodGVzdF9kcywgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoLCBzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICBjb2xsYXRlX2ZuPXBhcnRpYWwoY29sbGF0ZV9wb3NlX3RleHQsIHRva2VuaXplcj10b2tlbml6ZXIpKQogICAgcmVmcywgaHlwcywgdWlkcyA9IGRlY29kZV9hbGwobW9kZWwsIGxvYWRlciwgdG9rZW5pemVyLCBkZXZpY2UpCgogICAgbWV0cmljcyA9IGNvbXB1dGVfbWV0cmljcyhyZWZzLCBoeXBzKQogICAgZGl2ID0gZGl2ZXJzaXR5KFtub3JtKGgpIGZvciBoIGluIGh5cHNdKQogICAgcG9vbCA9IENvdW50ZXIobm9ybShyKSBmb3IgciBpbiByZWZzKQogICAgdHJpdmlhbCA9IFtwb29sLm1vc3RfY29tbW9uKDEpWzBdWzBdXSAqIGxlbihyZWZzKQogICAgdHJpdmlhbF9tZXRyaWNzID0gY29tcHV0ZV9tZXRyaWNzKHJlZnMsIHRyaXZpYWwpCiAgICBzZW5zID0gc2Vuc2l0aXZpdHkobW9kZWwsIGxvYWRlciwgdG9rZW5pemVyLCBkZXZpY2UpCgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDQyKQogICAgaWR4ID0gcm5nLmNob2ljZShsZW4ocmVmcyksIHNpemU9bWluKGFyZ3MuZXhhbXBsZXMsIGxlbihyZWZzKSksIHJlcGxhY2U9RmFsc2UpCiAgICBleGFtcGxlcyA9IFt7InVpZCI6IHVpZHNbaV0sICJyZWYiOiByZWZzW2ldLCAiaHlwIjogaHlwc1tpXX0gZm9yIGkgaW4gaWR4XQoKICAgIHJlcG9ydCA9IHsidGVzdF9tZXRyaWNzIjogbWV0cmljcywgInRyaXZpYWxfYmFzZWxpbmUiOiB0cml2aWFsX21ldHJpY3MsCiAgICAgICAgICAgICAgImRpdmVyc2l0eSI6IGRpdiwgInNlbnNpdGl2aXR5Ijogc2VucywgImV4YW1wbGVzIjogZXhhbXBsZXMsCiAgICAgICAgICAgICAgImJlYXRzX3RyaXZpYWxfYmxldTQiOiBtZXRyaWNzWyJCTEVVLTQiXSA+IHRyaXZpYWxfbWV0cmljc1siQkxFVS00Il0sCiAgICAgICAgICAgICAgInBvc2VfY29uZGl0aW9uZWQiOiBzZW5zWyJwb3NlX2NvbmRpdGlvbmVkIl0sCiAgICAgICAgICAgICAgImNvbGxhcHNlZCI6IGRpdlsidW5pcXVlX2ZyYWN0aW9uIl0gPCAwLjA1fQogICAgKG91dCAvICJldmFsX3JlcG9ydC5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTEpKQoKICAgIHByaW50KGpzb24uZHVtcHMoe2s6IHYgZm9yIGssIHYgaW4gcmVwb3J0Lml0ZW1zKCkKICAgICAgICAgICAgICAgICAgICAgIGlmIGsgbm90IGluICgiZXhhbXBsZXMiLCAic2Vuc2l0aXZpdHkiKX0sIGluZGVudD0xKSkKICAgIHByaW50KGYiXG4tLS0ge2xlbihleGFtcGxlcyl9IGhlbGQtb3V0IGV4YW1wbGVzIChyYW5kb20pIC0tLSIpCiAgICBmb3IgZSBpbiBleGFtcGxlczoKICAgICAgICBwcmludChmIlVJRD17ZVsndWlkJ119XG4gIFJFRjoge2VbJ3JlZiddfVxuICBIWVA6IHtlWydoeXAnXX0iKQogICAgcHJpbnQoIlxuLS0tIHNlbnNpdGl2aXR5IC0tLSIpCiAgICBmb3IgaSwgKGIsIHMsIHosIGYpIGluIGVudW1lcmF0ZSh6aXAoc2Vuc1siYmFzZSJdLCBzZW5zWyJzd2FwcGVkIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2Vuc1siemVyb2VkIl0sIHNlbnNbInNodWZmbGVkIl0pKToKICAgICAgICBwcmludChmIlt7aX1dIGJhc2U9e2Ihcn0gc3dhcD17cyFyfSB6ZXJvPXt6IXJ9IHNodWZmbGU9e2Yhcn0iKQogICAgcHJpbnQoZiJcbnBvc2VfY29uZGl0aW9uZWQ9e3NlbnNbJ3Bvc2VfY29uZGl0aW9uZWQnXX0gIgogICAgICAgICAgZiJjb2xsYXBzZWQ9e3JlcG9ydFsnY29sbGFwc2VkJ119IiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK"}, {"path": "README.md", "root": "POC", "sha256": "6b7fcba54c17c78e4e5605c9f22dd8853795b0d723d17dc9c958ad615573a31e", "b64": "IyBUNS1zbWFsbCBQb3NlLXRvLVRleHQgUE9DIChyZXNlYXJjaCBvbmx5IOKAlCBuZXZlciBzZXJ2aW5nKQoKIyMgUmV1c2VkIGV4aXN0aW5nIGNvZGUgKG5vIGR1cGxpY2F0aW9uKQotIGByZXNlYXJjaC90cmFpbmluZy9hdWRpdF9kYXRhc2V0LnB5YDogYGNsZWFuX2RhdGFmcmFtZWAsIGB2aWRlb19zcGxpdGAKICAob2ZmaWNpYWwgY2xlYW5pbmcgKyB2aWRlby1sZXZlbCA4MC8xMC8xMCBzcGxpdDsgc3Vic2V0IGtlZXBzIHdob2xlIHZpZGVvcykKLSBgcmVzZWFyY2gvdHJhaW5pbmcvdHJhaW5fc2lnbl90b190ZXh0LnB5YDogYHJlYWRfbm9ybWFsaXplZF9wb3NlYCwKICBgbG9hZF9wb3NlX2l0ZW1gLCBgUmVtb3RlUG9zZVNvdXJjZWAvYFJlbW90ZUNvbmNhdEZpbGVgLCBgcHJlZmxpZ2h0X3Bvc2Vfc291cmNlYAogIChjYW5vbmljYWwgYHBvc2Vfbm9ybWFsaXplX3YxYCwgTUFYX0ZSQU1FUz0xNjAsIFsxNjAsMTYyOV0gKyB0cnVlIGxlbmd0aCkKLSBgYmFja2VuZC9hcHAvbWwvcHJlcHJvY2Vzc2luZy9gOiBgbm9ybWFsaXplX3Bvc2Vfc2VxdWVuY2VgLAogIGByZXNpemVfc2VxdWVuY2VgLCBsYW5kbWFyayBjb25zdGFudHMgKDU0M8OXMyA9IDE2MjkpCi0gVjEgY29sbGFwc2UgcmVmZXJlbmNlOiBgcmVzZWFyY2gvZXZhbHVhdGlvbi9yZXBvcnRzLzM1Mjg2ODkyNC9gCiAgKGRpdmVyc2l0eSAwLjk5JSwgdG9wLXByZWQgNDQuOCUg4oaSIHRoZSBiYXIgdGhpcyBQT0MgbXVzdCBiZWF0KQoKIyMgVG9rZW5pemVyIGRlY2lzaW9uIChhcmNoaXRlY3R1cmFsKQpUaGUgZXhpc3RpbmcgMTZrIFNpZ25TYWdlIFNlbnRlbmNlUGllY2UgdW5pZ3JhbSB2b2NhYnVsYXJ5IGlzIElOQ09NUEFUSUJMRQp3aXRoIHByZXRyYWluZWQgVDU6IFQ1LXNtYWxsJ3MgZW1iZWRkaW5ncy9kZWNvZGVyIGFyZSB0cmFpbmVkIGZvciBpdHMgb3duCjMyayBTZW50ZW5jZVBpZWNlIHZvY2FidWxhcnkgKHBhZD0wLCBlb3M9MSwgdW5rPTIpLiBSZXVzaW5nIHRoZSAxNmsgdm9jYWIKd291bGQgZGlzY2FyZCBhbGwgcHJldHJhaW5lZCBsYW5ndWFnZSBrbm93bGVkZ2UuIFRoZXJlZm9yZSB0aGlzIFBPQyB1c2VzCnRoZSBvZmZpY2lhbCBgZ29vZ2xlLXQ1L3Q1LXNtYWxsYCB0b2tlbml6ZXIgKyBwcmV0cmFpbmVkIGRlY29kZXIsIGFuZCB0aGUKY2hlY2twb2ludCByZWNvcmRzIGB0b2tlbml6ZXI6IGdvb2dsZS10NS90NS1zbWFsbGAgZm9yIHJlcHJvZHVjaWJpbGl0eS4KCiMjIEZpbGVzCi0gYG1vZGVsLnB5YCDigJQgUG9zZVByb2plY3Rpb24gKyBQb3NlVDUgKGBpbnB1dHNfZW1iZWRzYCB3aXJpbmcpCi0gYG1hdGVyaWFsaXplLnB5YCDigJQgc21hbGwgdmlkZW8tbGV2ZWwgc3Vic2V0IOKGkiBmcDE2IGAubnB6YCBzaGFyZHMgKEthZ2dsZSkKLSBgdHJhaW4ucHlgIOKAlCBTdGFnZSBBIChmcm96ZW4gVDUpIC8gU3RhZ2UgQiAoam9pbnQpICsgc3ludGhldGljIHdpcmluZyBnYXRlCi0gYGV2YWx1YXRlLnB5YCDigJQgV0VSL0JMRVUtMS4uNC9ST1VHRS1ML0VNL2RpdmVyc2l0eS9iYXNlbGluZS9zZW5zaXRpdml0eQoKIyMgQ29tbWFuZHMKTG9jYWwgd2lyaW5nIGdhdGUgKENQVSwgbm8gZGF0YXNldCwgbm8gdG9rZW4pOgpgYmFja2VuZC8udmVudi9iaW4vcHl0aG9uIHJlc2VhcmNoL3Q1X3Bvc2VfcG9jL3RyYWluLnB5IC0tc3ludGhldGljLWdhdGUgLS1vdXQgLi90NV9wb2NfZ2F0ZWAKCkthZ2dsZSBHUFUgKG5lZWRzIEhGX1RPS0VOIHdpdGggaVNpZ24gYWNjZXNzKToKYHB5dGhvbiBtYXRlcmlhbGl6ZS5weSAtLWNzdiAvcGF0aC9pU2lnbl92MS4xLmNzdiAtLXBvc2Utc291cmNlIHJlbW90ZSAtLW91dCAuL3Q1X3BvY19zaGFyZHNgCmBweXRob24gdHJhaW4ucHkgLS1zaGFyZHMgLi90NV9wb2Nfc2hhcmRzIC0tb3V0IC4vdDVfcG9jXzVrIC0tZXBvY2hzIDhgCmBweXRob24gZXZhbHVhdGUucHkgLS1jaGVja3BvaW50IC4vdDVfcG9jXzVrL3Q1X3Bvc2VfcHJvamVjdGlvbi5wdCAtLXNoYXJkcyAuL3Q1X3BvY19zaGFyZHMgLS1vdXQgLi90NV9wb2NfZXZhbGAKCiMjIEV4dHJhIGRlcHMgKHJlc2VhcmNoIHZlbnYgb25seSwgTk9UIGFkZGVkIHRvIGJhY2tlbmQvcmVxdWlyZW1lbnRzLnR4dCkKYHRyYW5zZm9ybWVycyBodWdnaW5nZmFjZV9odWIgcG9zZS1mb3JtYXRgICgrIGBubHRrYCBmb3IgQkxFVS0xLi40KQo="}, {"path": "backend/app/__init__.py", "root": "SUP", "sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "b64": ""}, {"path": "backend/app/ml/models/__init__.py", "root": "SUP", "sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "b64": ""}, {"path": "backend/app/ml/models/sign_to_text.py", "root": "SUP", "sha256": "e7714e43fb9be98e31b3288cf8b5c00b6c38f92c099f83e551831c48f3afa592", "b64": "IiIiU2lnblNhZ2UgU2lnbi0+VGV4dCBtb2RlbCBkZWZpbml0aW9uLgoKVmVyYmF0aW0gcG9ydCBvZiB0aGUgdHJhaW5pbmcgbm90ZWJvb2sncyBGcmFtZVByb2plY3Rpb24vUG9zaXRpb25hbEVuY29kaW5nLwpTaWduVHJhbnNmb3JtZXIgY2xhc3NlcyAoc2lnbnNhZ2UuaXB5bmIsIFNlY3Rpb24gOCkuIG1heF9mcmFtZXMvbWF4X3Rva2Vucy8KcGFkX2lkeCBhcmUgZXhwbGljaXQgY29uc3RydWN0b3IgYXJndW1lbnRzIChub3QgaW1wbGljaXQgbm90ZWJvb2sgZ2xvYmFscyksCnNvIHRoaXMgY2xhc3MgaGFzIG5vIGRlcGVuZGVuY3kgb24gdHJhaW5pbmctbm90ZWJvb2sgc3RhdGUgYW5kIGNhbiBiZQpyZWNvbnN0cnVjdGVkIGZyb20gYSBjaGVja3BvaW50J3Mgb3duIG1vZGVsX2NvbmZpZyBpbiBhIGZyZXNoIHByb2Nlc3MuIFRoZQpmb3J3YXJkL2VuY29kZS9kZWNvZGUgY29tcHV0YXRpb24gaXRzZWxmIGlzIGJ5dGUtZm9yLWJ5dGUgdW5jaGFuZ2VkLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCgpjbGFzcyBGcmFtZVByb2plY3Rpb24obm4uTW9kdWxlKToKICAgICIiIlByb2plY3Qgc3BhdGlhbCBsYW5kbWFya3MgcGx1cyBleHBsaWNpdCBmcmFtZS10by1mcmFtZSBtb3Rpb24gZmVhdHVyZXMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlucHV0X2RpbSwgZF9tb2RlbCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5uZXQgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoaW5wdXRfZGltLCAzODQpLAogICAgICAgICAgICBubi5MYXllck5vcm0oMzg0KSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KDAuMTApLAogICAgICAgICAgICBubi5MaW5lYXIoMzg0LCBkX21vZGVsKSwKICAgICAgICAgICAgbm4uTGF5ZXJOb3JtKGRfbW9kZWwpLAogICAgICAgICkKICAgICAgICBzZWxmLnRlbXBvcmFsID0gbm4uQ29udjFkKGRfbW9kZWwsIGRfbW9kZWwsIGtlcm5lbF9zaXplPTUsIHBhZGRpbmc9MiwgZ3JvdXBzPWRfbW9kZWwpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgIyBFeHBsaWNpdCB2ZWxvY2l0eSBpcyB6ZXJvIGZvciB0aGUgZmlyc3QgZnJhbWUuIFRoaXMgbWFrZXMgdGhlIHRlbXBvcmFsCiAgICAgICAgIyBzaWduYWwgYXZhaWxhYmxlIGJlZm9yZSBhdHRlbnRpb24gcmF0aGVyIHRoYW4gYXNraW5nIHRoZSBlbmNvZGVyIHRvCiAgICAgICAgIyBpbmZlciBhbGwgZGVyaXZhdGl2ZXMgZnJvbSBhYnNvbHV0ZSBjb29yZGluYXRlcyBhbG9uZS4KICAgICAgICBkZWx0YSA9IHRvcmNoLnplcm9zX2xpa2UoeCkKICAgICAgICBkZWx0YVs6LCAxOl0gPSB4WzosIDE6XSAtIHhbOiwgOi0xXQogICAgICAgIHkgPSBzZWxmLm5ldCh0b3JjaC5jYXQoW3gsIGRlbHRhXSwgZGltPS0xKSkKICAgICAgICB5ID0gc2VsZi50ZW1wb3JhbCh5LnRyYW5zcG9zZSgxLCAyKSkudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgcmV0dXJuIHkKCgpjbGFzcyBQb3NpdGlvbmFsRW5jb2Rpbmcobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkX21vZGVsLCBtYXhfbGVuKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBwZSA9IHRvcmNoLnplcm9zKG1heF9sZW4sIGRfbW9kZWwpCiAgICAgICAgcG9zID0gdG9yY2guYXJhbmdlKG1heF9sZW4sIGR0eXBlPXRvcmNoLmZsb2F0MzIpLnVuc3F1ZWV6ZSgxKQogICAgICAgIGRpdiA9IHRvcmNoLmV4cCh0b3JjaC5hcmFuZ2UoMCwgZF9tb2RlbCwgMiwgZHR5cGU9dG9yY2guZmxvYXQzMikgKiAoLW5wLmxvZygxMDAwMC4wKSAvIGRfbW9kZWwpKQogICAgICAgIHBlWzosIDA6OjJdID0gdG9yY2guc2luKHBvcyAqIGRpdikKICAgICAgICBwZVs6LCAxOjoyXSA9IHRvcmNoLmNvcyhwb3MgKiBkaXYpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoInBlIiwgcGUudW5zcXVlZXplKDApKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiB4ICsgc2VsZi5wZVs6LCA6IHguc2l6ZSgxKV0KCgpjbGFzcyBTaWduVHJhbnNmb3JtZXIobm4uTW9kdWxlKToKICAgICIiIlNwYXRpYWwtdGVtcG9yYWwgVHJhbnNmb3JtZXIgZm9yIFNpZ24tPlRleHQuCgogICAgU2FtZSBhcmNoaXRlY3R1cmUgYXMgdHJhaW5pbmcuIG1heF9mcmFtZXMvbWF4X3Rva2Vucy9wYWRfaWR4IGFyZQogICAgZXhwbGljaXQgY29uc3RydWN0b3IgYXJndW1lbnRzIChwcmV2aW91c2x5IHRoZSBub3RlYm9vayBnbG9iYWxzCiAgICBNQVhfRlJBTUVTLCBNQVhfVE9LRU5TLCBQQURfSURYKSBzbyB0aGlzIGNsYXNzIGhhcyBubyBleHRlcm5hbCBzdGF0ZQogICAgZGVwZW5kZW5jeS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0X2RpbSwKICAgICAgICB2b2NhYl9zaXplLAogICAgICAgIG1heF9mcmFtZXMsCiAgICAgICAgbWF4X3Rva2VucywKICAgICAgICBwYWRfaWR4LAogICAgICAgIGRfbW9kZWw9MTkyLAogICAgICAgIG5oZWFkPTQsCiAgICAgICAgbGF5ZXJzPTMsCiAgICAgICAgZmY9NTEyLAogICAgICAgIGRyb3BvdXQ9MC4xNSwKICAgICAgICBub3JtX2ZpcnN0PUZhbHNlLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnBhZF9pZHggPSBwYWRfaWR4CiAgICAgICAgc2VsZi5ub3JtX2ZpcnN0ID0gbm9ybV9maXJzdAogICAgICAgIHNlbGYuaW5wdXRfcHJvaiA9IEZyYW1lUHJvamVjdGlvbihpbnB1dF9kaW0sIGRfbW9kZWwpCiAgICAgICAgc2VsZi5zcmNfcG9zID0gUG9zaXRpb25hbEVuY29kaW5nKGRfbW9kZWwsIG1heF9mcmFtZXMgKyA0KQogICAgICAgIHNlbGYudG9rX2VtYmVkID0gbm4uRW1iZWRkaW5nKHZvY2FiX3NpemUsIGRfbW9kZWwsIHBhZGRpbmdfaWR4PXBhZF9pZHgpCiAgICAgICAgc2VsZi50Z3RfcG9zID0gUG9zaXRpb25hbEVuY29kaW5nKGRfbW9kZWwsIG1heF90b2tlbnMgKyA0KQogICAgICAgIGVuYyA9IG5uLlRyYW5zZm9ybWVyRW5jb2RlckxheWVyKGRfbW9kZWwsIG5oZWFkLCBmZiwgZHJvcG91dCwgYmF0Y2hfZmlyc3Q9VHJ1ZSwgbm9ybV9maXJzdD1ub3JtX2ZpcnN0KQogICAgICAgIGRlYyA9IG5uLlRyYW5zZm9ybWVyRGVjb2RlckxheWVyKGRfbW9kZWwsIG5oZWFkLCBmZiwgZHJvcG91dCwgYmF0Y2hfZmlyc3Q9VHJ1ZSwgbm9ybV9maXJzdD1ub3JtX2ZpcnN0KQogICAgICAgIHNlbGYuZW5jb2RlciA9IG5uLlRyYW5zZm9ybWVyRW5jb2RlcihlbmMsIGxheWVycykKICAgICAgICBzZWxmLmRlY29kZXIgPSBubi5UcmFuc2Zvcm1lckRlY29kZXIoZGVjLCBsYXllcnMpCiAgICAgICAgc2VsZi5vdXQgPSBubi5MaW5lYXIoZF9tb2RlbCwgdm9jYWJfc2l6ZSkKICAgICAgICBzZWxmLnNjYWxlID0gZF9tb2RlbCAqKiAwLjUKCiAgICBkZWYgZW5jb2RlKHNlbGYsIGZyYW1lcywgc3JjX3BhZCk6CiAgICAgICAgcmV0dXJuIHNlbGYuZW5jb2RlcihzZWxmLnNyY19wb3Moc2VsZi5pbnB1dF9wcm9qKGZyYW1lcykpLCBzcmNfa2V5X3BhZGRpbmdfbWFzaz1zcmNfcGFkKQoKICAgIGRlZiBkZWNvZGUoc2VsZiwgdG9rZW5zLCBtZW1vcnksIHRndF9tYXNrLCBzcmNfcGFkLCB0Z3RfcGFkPU5vbmUpOgogICAgICAgIHggPSBzZWxmLnRndF9wb3Moc2VsZi50b2tfZW1iZWQodG9rZW5zKSAqIHNlbGYuc2NhbGUpCiAgICAgICAgcmV0dXJuIHNlbGYub3V0KAogICAgICAgICAgICBzZWxmLmRlY29kZXIoCiAgICAgICAgICAgICAgICB4LCBtZW1vcnksIHRndF9tYXNrPXRndF9tYXNrLAogICAgICAgICAgICAgICAgdGd0X2tleV9wYWRkaW5nX21hc2s9dGd0X3BhZCwKICAgICAgICAgICAgICAgIG1lbW9yeV9rZXlfcGFkZGluZ19tYXNrPXNyY19wYWQsCiAgICAgICAgICAgICkKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgZnJhbWVzLCBzcmNfcGFkLCB0b2tlbnMsIHRndF9tYXNrKToKICAgICAgICByZXR1cm4gc2VsZi5kZWNvZGUoCiAgICAgICAgICAgIHRva2Vucywgc2VsZi5lbmNvZGUoZnJhbWVzLCBzcmNfcGFkKSwgdGd0X21hc2ssIHNyY19wYWQsCiAgICAgICAgICAgIHRndF9wYWQ9dG9rZW5zLmVxKHNlbGYucGFkX2lkeCksCiAgICAgICAgKQoKCmRlZiBwYWRkaW5nX21hc2sobGVuZ3RocywgbWF4X2xlbik6CiAgICByZXR1cm4gdG9yY2guYXJhbmdlKG1heF9sZW4sIGRldmljZT1sZW5ndGhzLmRldmljZSkudW5zcXVlZXplKDApID49IGxlbmd0aHMudW5zcXVlZXplKDEpCgoKZGVmIGNhdXNhbF9tYXNrKG4sIGRldmljZSk6CiAgICByZXR1cm4gdG9yY2gudHJpdSh0b3JjaC5vbmVzKG4sIG4sIGRldmljZT1kZXZpY2UsIGR0eXBlPXRvcmNoLmJvb2wpLCBkaWFnb25hbD0xKQo="}, {"path": "backend/app/ml/preprocessing/__init__.py", "root": "SUP", "sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "b64": ""}, {"path": "backend/app/ml/preprocessing/landmarks.py", "root": "SUP", "sha256": "47dd4083e4ca26ce3022e7245e0df3211c6aea3f76ae5189ddae128966773905", "b64": "IiIiTGFuZG1hcmsgbGF5b3V0IGNvbnN0YW50cyBmb3IgdGhlIFNpZ24tPlRleHQgcGlwZWxpbmUuCgpUaGVzZSBudW1iZXJzIGNvbWUgZGlyZWN0bHkgZnJvbSBsYW5kbWFya19jb250cmFjdC5qc29uIChjb21wb25lbnRfb3JkZXIgLwpjb21wb25lbnRfcmFuZ2VzKSwgd2hpY2ggaXMgaXRzZWxmIGRlcml2ZWQgZnJvbSB0aGUgcmVhbCBNZWRpYVBpcGUgSG9saXN0aWMKcG9zZS1maWxlIGhlYWRlciBhdCB0cmFpbmluZyB0aW1lIChzaWduc2FnZS5pcHluYiwgU2VjdGlvbiA1YikuIERvIG5vdApjaGFuZ2UgdGhlc2Ugd2l0aG91dCByZWdlbmVyYXRpbmcgdGhhdCBjb250cmFjdCBhbmQgcmUtdmFsaWRhdGluZyAoc2VlCm1sL2NvbnRyYWN0cy9sYW5kbWFya19jb250cmFjdC5weSkgLS0gdGhpcyBtb2R1bGUgbmV2ZXIgaW52ZW50cyBvcmRlcmluZywKaXQganVzdCBuYW1lcyB0aGUgY29uc3RhbnRzIHRoZSB0cmFpbmVkIG1vZGVsIGFscmVhZHkgYXNzdW1lcy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKClRPVEFMX0xBTkRNQVJLUyA9IDU0MwpDT09SRElOQVRFU19QRVJfTEFORE1BUksgPSAzClJBV19GRUFUVVJFX0RJTSA9IFRPVEFMX0xBTkRNQVJLUyAqIENPT1JESU5BVEVTX1BFUl9MQU5ETUFSSyAgIyAxNjI5CgojIFRoZSBtb2RlbCdzIEZyYW1lUHJvamVjdGlvbiBjb25jYXRlbmF0ZXMgZWFjaCBmcmFtZSB3aXRoIGl0cyBvd24KIyBmcmFtZS10by1mcmFtZSBkZWx0YSBiZWZvcmUgaXRzIGZpcnN0IGxpbmVhciBsYXllciAtLSBzZWUKIyBtbC9tb2RlbHMvc2lnbl90b190ZXh0LnB5OkZyYW1lUHJvamVjdGlvbi4gUHJlcHJvY2Vzc2luZyAodGhpcyBwYWNrYWdlKQojIG11c3QgTk9UIHByZWNvbXB1dGUgdGhhdCBkZWx0YTsgaXQgc2hpcHMgcGxhaW4gbm9ybWFsaXplZCAxNjI5LUQgZnJhbWVzCiMgYW5kIHRoZSBtb2RlbCBwcm9kdWNlcyB0aGUgMzI1OC1EIHByb2plY3Rpb24gaW5wdXQgaW50ZXJuYWxseS4KTU9ERUxfSU5QVVRfRElNID0gUkFXX0ZFQVRVUkVfRElNICogMiAgIyAzMjU4CgojIE1lZGlhUGlwZSBwb3NlIGxhbmRtYXJrczogZ2xvYmFsIGluZGV4IDIzID0gbGVmdCBoaXAsIDI0ID0gcmlnaHQgaGlwLgojIG5vcm1hbGl6ZV9wb3NlX3NlcXVlbmNlKCkgcm9vdC1jZW50ZXJzIG9uIHRoZSBtaWRwb2ludCBvZiB0aGVzZSB0d28gcG9pbnRzLgpISVBfTEFORE1BUktfSU5ESUNFUyA9ICgyMywgMjQpCkJPRFlfTEFORE1BUktfQ09VTlQgPSAzMyAgIyBmaXJzdCAzMyBnbG9iYWwgaW5kaWNlcyA9IHRoZSAicG9zZSIgY29tcG9uZW50CgojIEJ1bXAgdGhpcyB3aGVuZXZlciBub3JtYWxpemVfcG9zZV9zZXF1ZW5jZSgpJ3MgYWxnb3JpdGhtIGNoYW5nZXMuCiMgbWwvY29udHJhY3RzL21vZGVsX2NvbnRyYWN0LnB5IGFzc2VydHMgdGhlIGRlcGxveW1lbnQgbWFuaWZlc3QncyBkZWNsYXJlZAojIG5vcm1hbGl6YXRpb25fdmVyc2lvbiBlcXVhbHMgdGhpcyBjb25zdGFudCwgc28gYSBzdGFsZSBidW5kbGUgcGFpcmVkIHdpdGgKIyBuZXcgcHJlcHJvY2Vzc2luZyBjb2RlIChvciB2aWNlIHZlcnNhKSBpcyBjYXVnaHQgYXQgbG9hZCB0aW1lLgpOT1JNQUxJWkFUSU9OX1ZFUlNJT04gPSAicG9zZV9ub3JtYWxpemVfdjEiCgoKZGVmIGFzc2VydF9sYW5kbWFya19zaGFwZShhcnIpIC0+IE5vbmU6CiAgICAiIiJSYWlzZSBWYWx1ZUVycm9yIGlmIGBhcnJgIGlzIG5vdCBzaGFwZWQgKFQsIDU0MywgMykuIiIiCiAgICBpZiBhcnIubmRpbSAhPSAzIG9yIGFyci5zaGFwZVsxOl0gIT0gKFRPVEFMX0xBTkRNQVJLUywgQ09PUkRJTkFURVNfUEVSX0xBTkRNQVJLKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIkV4cGVjdGVkIGxhbmRtYXJrcyBzaGFwZWQgKFQsIHtUT1RBTF9MQU5ETUFSS1N9LCB7Q09PUkRJTkFURVNfUEVSX0xBTkRNQVJLfSksICIKICAgICAgICAgICAgZiJyZWNlaXZlZCB7YXJyLnNoYXBlfSIKICAgICAgICApCg=="}, {"path": "backend/app/ml/preprocessing/normalization.py", "root": "SUP", "sha256": "125b3419fac72a69cb58053d4cea927e43857db4503fc6b41064eff25a650ea0", "b64": "IiIiUG9zZSBub3JtYWxpemF0aW9uIC0tIHZlcmJhdGltIHBvcnQgb2YgdGhlIHRyYWluaW5nIG5vdGVib29rJ3MKbm9ybWFsaXplX3Bvc2Vfc2VxdWVuY2UoKSAoc2lnbnNhZ2UuaXB5bmIsIFNlY3Rpb24gNWEgLyB0aGUgbm90ZWJvb2sncyBvd24KZXhwb3J0ZWQgc2lnbnNhZ2VfaW5mZXJlbmNlL3ByZXByb2Nlc3NpbmcucHkpLgoKUm9vdC1jZW50ZXJzIGV2ZXJ5IGZyYW1lIG9uIHRoZSBoaXAgbWlkcG9pbnQgYW5kIHNjYWxlcyBieSB0aGUgcGVyLXNlcXVlbmNlCmJvZHktbGFuZG1hcmsgc3ByZWFkLiBUaGlzIE1VU1Qgc3RheSBieXRlLWZvci1ieXRlIGlkZW50aWNhbCB0byB0aGUKdHJhaW5pbmctdGltZSBpbXBsZW1lbnRhdGlvbiwgb3IgbGl2ZSBpbmZlcmVuY2Ugd2lsbCBzaWxlbnRseSBkaXZlcmdlIGZyb20Kd2hhdCB0aGUgbW9kZWwgd2FzIGFjdHVhbGx5IHRyYWluZWQgb24uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgd2FybmluZ3MKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAubGFuZG1hcmtzIGltcG9ydCAoCiAgICBCT0RZX0xBTkRNQVJLX0NPVU5ULAogICAgQ09PUkRJTkFURVNfUEVSX0xBTkRNQVJLLAogICAgSElQX0xBTkRNQVJLX0lORElDRVMsCiAgICBUT1RBTF9MQU5ETUFSS1MsCikKCgpkZWYgbm9ybWFsaXplX3Bvc2Vfc2VxdWVuY2UoZmxhdDogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICIiIlJvb3QtY2VudGVyIGFuZCBzY2FsZSBvbmUgcG9zZSBzZXF1ZW5jZS4KCiAgICBBcmdzOgogICAgICAgIGZsYXQ6IChmcmFtZXMsIDU0MyozKSByYXcgbGFuZG1hcmsgY29vcmRpbmF0ZXMuCiAgICBSZXR1cm5zOgogICAgICAgIGZsb2F0MzIgKGZyYW1lcywgNTQzKjMpIG5vcm1hbGl6ZWQgY29vcmRpbmF0ZXMuCiAgICAiIiIKICAgIHggPSBmbGF0LnJlc2hhcGUoZmxhdC5zaGFwZVswXSwgVE9UQUxfTEFORE1BUktTLCBDT09SRElOQVRFU19QRVJfTEFORE1BUkspLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIGhpcCA9IHhbOiwgbGlzdChISVBfTEFORE1BUktfSU5ESUNFUyksIDpdCiAgICB3aXRoIHdhcm5pbmdzLmNhdGNoX3dhcm5pbmdzKCk6CiAgICAgICAgd2FybmluZ3MuZmlsdGVyd2FybmluZ3MoImlnbm9yZSIsIG1lc3NhZ2U9Ik1lYW4gb2YgZW1wdHkgc2xpY2UiKQogICAgICAgIGNlbnRlciA9IG5wLm5hbm1lYW4oaGlwLCBheGlzPTEsIGtlZXBkaW1zPVRydWUpCgogICAgYmFkX2NlbnRlciA9IH5ucC5pc2Zpbml0ZShjZW50ZXIpLmFsbChheGlzPSgxLCAyKSkKICAgIGlmIGJhZF9jZW50ZXIuYW55KCk6CiAgICAgICAgd2l0aCB3YXJuaW5ncy5jYXRjaF93YXJuaW5ncygpOgogICAgICAgICAgICB3YXJuaW5ncy5maWx0ZXJ3YXJuaW5ncygiaWdub3JlIiwgbWVzc2FnZT0iTWVhbiBvZiBlbXB0eSBzbGljZSIpCiAgICAgICAgICAgIGZhbGxiYWNrX2NlbnRlciA9IG5wLm5hbm1lYW4oeFtiYWRfY2VudGVyLCA6Qk9EWV9MQU5ETUFSS19DT1VOVCwgOl0sIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgICAgICBjZW50ZXJbYmFkX2NlbnRlcl0gPSBmYWxsYmFja19jZW50ZXIKCiAgICB4ID0geCAtIGNlbnRlcgoKICAgIGJvZHkgPSB4WzosIDpCT0RZX0xBTkRNQVJLX0NPVU5ULCA6XQogICAgd2l0aCB3YXJuaW5ncy5jYXRjaF93YXJuaW5ncygpOgogICAgICAgIHdhcm5pbmdzLmZpbHRlcndhcm5pbmdzKCJpZ25vcmUiLCBtZXNzYWdlPSJEZWdyZWVzIG9mIGZyZWVkb20iKQogICAgICAgIHNjYWxlX3Blcl9mcmFtZSA9IG5wLm5hbnN0ZChib2R5LnJlc2hhcGUoYm9keS5zaGFwZVswXSwgLTEpLCBheGlzPTEpCgogICAgdmFsaWRfc2NhbGUgPSBzY2FsZV9wZXJfZnJhbWVbbnAuaXNmaW5pdGUoc2NhbGVfcGVyX2ZyYW1lKSAmIChzY2FsZV9wZXJfZnJhbWUgPiAxZS02KV0KICAgIGlmIHZhbGlkX3NjYWxlLnNpemUgPT0gMDoKICAgICAgICBzY2FsZSA9IDEuMAogICAgZWxzZToKICAgICAgICBzY2FsZSA9IG5wLm5hbm1lZGlhbih2YWxpZF9zY2FsZSkKICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUoc2NhbGUpIG9yIHNjYWxlIDw9IDFlLTY6CiAgICAgICAgICAgIHNjYWxlID0gMS4wCgogICAgeCA9IHggLyBmbG9hdChzY2FsZSkKICAgIHggPSBucC5uYW5fdG9fbnVtKHgsIG5hbj0wLjAsIHBvc2luZj0wLjAsIG5lZ2luZj0wLjApCiAgICByZXR1cm4geC5yZXNoYXBlKHguc2hhcGVbMF0sIC0xKS5hc3R5cGUobnAuZmxvYXQzMikK"}, {"path": "backend/app/ml/preprocessing/sequence.py", "root": "SUP", "sha256": "132fdceac622c995a26191088c7d2a151055adc7bbdf5bb38e086cdcf86104dc", "b64": "IiIiVGVtcG9yYWwgcmVzaXppbmcgYW5kIGxpdmUtZnJhbWUgcHJlcGFyYXRpb24gLS0gdmVyYmF0aW0gcG9ydCBvZiB0aGUKdHJhaW5pbmcgbm90ZWJvb2sncyByZXNpemVfc2VxdWVuY2UoKS9wcmVwYXJlX2xpdmVfbGFuZG1hcmtzKCkvCmxpdmVfbW90aW9uX2VuZXJneSgpIChzaWduc2FnZS5pcHluYiwgU2VjdGlvbiA1YSAmIFNlY3Rpb24gMTUpLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC5sYW5kbWFya3MgaW1wb3J0IGFzc2VydF9sYW5kbWFya19zaGFwZQpmcm9tIC5ub3JtYWxpemF0aW9uIGltcG9ydCBub3JtYWxpemVfcG9zZV9zZXF1ZW5jZQoKCmRlZiByZXNpemVfc2VxdWVuY2Uoc2VxOiBucC5uZGFycmF5LCB0YXJnZXRfbGVuOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJVbmlmb3JtbHkgZG93bnNhbXBsZSBsb25nIHNlcXVlbmNlczsgemVyby1wYWQgc2hvcnQgb25lcy4iIiIKICAgIHNyY19sZW4gPSBzZXEuc2hhcGVbMF0KICAgIGlmIHNyY19sZW4gPD0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFbXB0eSBwb3NlIHNlcXVlbmNlIikKICAgIGlmIHNyY19sZW4gPT0gdGFyZ2V0X2xlbjoKICAgICAgICByZXR1cm4gc2VxLmFzdHlwZShucC5mbG9hdDMyLCBjb3B5PUZhbHNlKQogICAgaWYgc3JjX2xlbiA+IHRhcmdldF9sZW46CiAgICAgICAgaWR4ID0gbnAubGluc3BhY2UoMCwgc3JjX2xlbiAtIDEsIHRhcmdldF9sZW4pCiAgICAgICAgcmV0dXJuIHNlcVtucC5yb3VuZChpZHgpLmFzdHlwZShucC5pbnQ2NCldLmFzdHlwZShucC5mbG9hdDMyLCBjb3B5PUZhbHNlKQogICAgb3V0ID0gbnAuemVyb3MoKHRhcmdldF9sZW4sIHNlcS5zaGFwZVsxXSksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBvdXRbOnNyY19sZW5dID0gc2VxCiAgICByZXR1cm4gb3V0CgoKZGVmIHByZXBhcmVfbGl2ZV9sYW5kbWFya3MobGFuZG1hcmtzX3h5ejogbnAubmRhcnJheSwgbWF4X2ZyYW1lczogaW50KSAtPiB0dXBsZVtucC5uZGFycmF5LCBpbnRdOgogICAgIiIiQ29udmVydCByYXcgKFQsIDU0MywgMykgY2FtZXJhIGxhbmRtYXJrcyBpbnRvIHRoZSBtb2RlbCdzIGlucHV0IHNoYXBlLgoKICAgIFJldHVybnMgKHByb2Nlc3NlZCwgdHJ1ZV9sZW4pOiBwcm9jZXNzZWQgaXMgKG1heF9mcmFtZXMsIDE2MjkpIGZsb2F0MzIsCiAgICB0cnVlX2xlbiBpcyBtaW4oVCwgbWF4X2ZyYW1lcykgLS0gdGhlIHJlYWwgKHVucGFkZGVkKSBzZXF1ZW5jZSBsZW5ndGggdGhlCiAgICBtb2RlbCBuZWVkcyBmb3IgaXRzIHBhZGRpbmcgbWFzay4gQ2FsbGVycyBtdXN0IE5PVCBwcmVjb21wdXRlIGRlbHRhczsKICAgIHRoZSBtb2RlbCBkb2VzIHRoYXQgaW50ZXJuYWxseSAoc2VlIG1sL21vZGVscy9zaWduX3RvX3RleHQucHkpLgogICAgIiIiCiAgICBhcnIgPSBucC5hc2FycmF5KGxhbmRtYXJrc194eXosIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBhc3NlcnRfbGFuZG1hcmtfc2hhcGUoYXJyKQogICAgaWYgbGVuKGFycikgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk5lZWQgYXQgbGVhc3QgdHdvIGZyYW1lcy4iKQogICAgZmxhdCA9IGFyci5yZXNoYXBlKGxlbihhcnIpLCAtMSkKICAgIG5vcm0gPSBub3JtYWxpemVfcG9zZV9zZXF1ZW5jZShmbGF0KQogICAgdHJ1ZV9sZW4gPSBtaW4obGVuKG5vcm0pLCBtYXhfZnJhbWVzKQogICAgcmV0dXJuIHJlc2l6ZV9zZXF1ZW5jZShub3JtLCBtYXhfZnJhbWVzKSwgdHJ1ZV9sZW4KCgpkZWYgY29tcHV0ZV9tb3Rpb25fZW5lcmd5KGxhbmRtYXJrc194eXo6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgIiIiTWVhbiBhYnNvbHV0ZSBmcmFtZS10by1mcmFtZSBkZWx0YSBvbiBub3JtYWxpemVkIGxhbmRtYXJrcy4KCiAgICBVc2VkIGFzIGEgY2hlYXAgImlzIGFueXRoaW5nIGFjdHVhbGx5IG1vdmluZyIgZ2F0ZSBiZWZvcmUgdHJ1c3RpbmcgdGhlCiAgICBtb2RlbCdzIG91dHB1dCAtLSBhIHN0YXRpYy9uby1zaWduaW5nIGZyYW1lIHNlcXVlbmNlIHNob3VsZCBub3QgcHJvZHVjZQogICAgYSBjb25maWRlbnQtbG9va2luZyB0cmFuc2xhdGlvbi4KICAgICIiIgogICAgYXJyID0gbnAuYXNhcnJheShsYW5kbWFya3NfeHl6LCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgbGVuKGFycikgPCAyOgogICAgICAgIHJldHVybiAwLjAKICAgIG5vcm0gPSBub3JtYWxpemVfcG9zZV9zZXF1ZW5jZShhcnIucmVzaGFwZShsZW4oYXJyKSwgLTEpKQogICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4obnAuYWJzKG5wLmRpZmYobm9ybSwgYXhpcz0wKSkpKQo="}, {"path": "research/training/audit_dataset.py", "root": "SUP", "sha256": "7dbd055e7ea51fd58bffcbc750e43ac409ce2e99e2c52f3b3f2a5b7fc1abd8c5", "b64": "IiIiU2lnblNhZ2UgZGF0YXNldCBpbnRlZ3JpdHkgYXVkaXQgKGlTaWduIHYxLjEsIHBvc2UtPkVuZ2xpc2ggdHJhbnNsYXRpb24gdGFzaykuCgpSZXByb2R1Y2VzIHRoZSB0cmFpbmluZyBub3RlYm9vaydzIGNsZWFuaW5nICsgdmlkZW8tbGV2ZWwgc3BsaXQgcG9saWN5CihyZXNlYXJjaC9ub3RlYm9va3Mvc2lnbnNhZ2Uvc2lnbnNhZ2UuaXB5bmIsIGNlbGxzIDYvMTEvMTIvMTMpIGFzIGFuCmltcG9ydGFibGUsIGRldGVybWluaXN0aWMgc2NyaXB0LCB0aGVuIHJlcG9ydHMgaW50ZWdyaXR5IHN0YXRpc3RpY3MgYW5kCndyaXRlcyBhIEpTT04gcmVwb3J0LgoKQ1NWLWxldmVsIGNoZWNrcyBydW4gYW55d2hlcmUgKHBhbmRhcyArIG51bXB5IG9ubHkpLiBQb3NlLWxldmVsIGNoZWNrcwooZGltZW5zaW9ucywgTmFOIHJhdGVzLCBmcmFtZSBjb3VudHMsIG1pc3NpbmcgbWVtYmVycykgbmVlZCBgcG9zZV9mb3JtYXRgCmFuZCBlaXRoZXIgYSBkaXJlY3Rvcnkgb2YgVUlELWxldmVsIC5wb3NlIGZpbGVzIG9yIHRoZSByZW1vdGUgWklQOwp0aGV5IGFyZSBza2lwcGVkIGdyYWNlZnVsbHkgd2hlbiB1bmF2YWlsYWJsZS4KClVzYWdlIChLYWdnbGUsIHdoZXJlIHRoZSBkYXRhIGxpdmVzKToKICAgIHB5dGhvbiBhdWRpdF9kYXRhc2V0LnB5IC0tY3N2IC9rYWdnbGUvd29ya2luZy9pc2lnbi9pU2lnbl92MS4xLmNzdiBcCiAgICAgICAgLS1wb3NlLXppcCA8cmVtb3RlIHppcCB1cmwgb3IgbG9jYWwgcGF0aD4gLS1zYW1wbGUgMjAwIFwKICAgICAgICAtLW91dCAva2FnZ2xlL3dvcmtpbmcvc2lnbnNhZ2Vfb3V0cHV0cy9kYXRhc2V0X2F1ZGl0Lmpzb24KCkxvY2FsIHNlbGYtdGVzdCAobm8gZGF0YSBuZWVkZWQpOgogICAgcHl0aG9uIGF1ZGl0X2RhdGFzZXQucHkgLS1zZWxmLXRlc3QKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHppcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKU0VFRCA9IDQyCk1BWF9UT0tFTlMgPSA5NiAgIyBtdXN0IG1hdGNoIHRoZSB0cmFpbmluZyB0b2tlbml6ZXIgcG9saWN5IChCT1MrRU9TKzw9OTQpCkVYUEVDVEVEX0NPTVBPTkVOVFMgPSAoKCJQT1NFIiwgMzMpLCAoIkZBQ0UiLCA0NjgpLCAoIkxFRlRfSEFORCIsIDIxKSwgKCJSSUdIVF9IQU5EIiwgMjEpKQoKCmRlZiBub3JtYWxpemVfdGV4dCh0ZXh0KSAtPiBzdHI6CiAgICAiIiJTYW1lIGFzIHRoZSBub3RlYm9vazogbG93ZXJjYXNlICsgY29sbGFwc2Ugd2hpdGVzcGFjZS4iIiIKICAgIHJldHVybiAiICIuam9pbihzdHIodGV4dCkubG93ZXIoKS5zdHJpcCgpLnNwbGl0KCkpCgoKZGVmIGNsZWFuX2RhdGFmcmFtZShkZjogcGQuRGF0YUZyYW1lKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIGRpY3RdOgogICAgIiIiQXBwbHkgdGhlIG5vdGVib29rJ3MgY2xlYW5pbmcgcG9saWN5OyByZXR1cm4gKGNsZWFuZWQsIHN0YXRzKS4iIiIKICAgIHN0YXRzOiBkaWN0ID0geyJyYXdfcm93cyI6IGludChsZW4oZGYpKX0KICAgIGRmID0gZGYuY29weSgpCiAgICBkZlsidWlkIl0gPSBkZlsidWlkIl0uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkKICAgIGRmWyJ0ZXh0Il0gPSBkZlsidGV4dCJdLmZpbGxuYSgiIikuYXN0eXBlKHN0cikuc3RyLnJlcGxhY2UociJccysiLCAiICIsIHJlZ2V4PVRydWUpLnN0ci5zdHJpcCgpCgogICAgcGFydHMgPSBkZlsidWlkIl0uc3RyLnJzcGxpdCgiLSIsIG49MSwgZXhwYW5kPVRydWUpCiAgICBkZlsidmlkZW9faWQiXSA9IHBhcnRzWzBdLndoZXJlKHBhcnRzWzFdLm5vdG5hKCkgJiAocGFydHNbMF0uc3RyLnN0cmlwKCkgIT0gIiIpLCBOb25lKQogICAgZGZbInNlcSJdID0gcGFydHNbMV0ud2hlcmUocGFydHNbMV0ubm90bmEoKSAmIChwYXJ0c1sxXS5zdHIuc3RyaXAoKSAhPSAiIiksIE5vbmUpCiAgICBtYWxmb3JtZWQgPSBkZlsidmlkZW9faWQiXS5pc25hKCkgfCBkZlsic2VxIl0uaXNuYSgpCiAgICBzdGF0c1sibWFsZm9ybWVkX3VpZCJdID0gaW50KG1hbGZvcm1lZC5zdW0oKSkKICAgIGRmID0gZGYubG9jW35tYWxmb3JtZWRdLmNvcHkoKQogICAgZGZbInZpZGVvX2lkIl0gPSBkZlsidmlkZW9faWQiXS5hc3R5cGUoc3RyKS5zdHIuc3RyaXAoKQogICAgZGZbInNlcSJdID0gZGZbInNlcSJdLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpCiAgICBkZlsibl93b3JkcyJdID0gZGZbInRleHQiXS5zdHIuc3BsaXQoKS5zdHIubGVuKCkKICAgIGRmWyJ0ZXh0X25vcm0iXSA9IGRmWyJ0ZXh0Il0ubWFwKG5vcm1hbGl6ZV90ZXh0KQoKICAgIGp1bmsgPSAoCiAgICAgICAgZGZbInRleHRfbm9ybSJdLmVxKCIiKQogICAgICAgIHwgZGZbInRleHRfbm9ybSJdLnN0ci5mdWxsbWF0Y2gociJcZCsiKQogICAgICAgIHwgZGZbInRleHRfbm9ybSJdLnN0ci5mdWxsbWF0Y2gociJbYS16XSIpCiAgICAgICAgfCBkZlsidGV4dF9ub3JtIl0uc3RyLmZ1bGxtYXRjaChyIlxbW15cXV0rXF0iKQogICAgICAgIHwgZGZbInRleHRfbm9ybSJdLnN0ci5mdWxsbWF0Y2gociJbXmEtejAtOV0rIikKICAgICkKICAgIGp1bmsgJj0gfmRmWyJ0ZXh0X25vcm0iXS5lcSgiaSIpICAjICJpIiBpcyBhIGxlZ2l0aW1hdGUgcHJvbm91bgogICAganVuayB8PSBkZlsidWlkIl0uZXEoIiIpCiAgICBzdGF0c1sianVua19yb3dzIl0gPSBpbnQoanVuay5zdW0oKSkKICAgIHN0YXRzWyJqdW5rX2JyZWFrZG93biJdID0gewogICAgICAgICJlbXB0eV90ZXh0IjogaW50KGRmWyJ0ZXh0X25vcm0iXS5lcSgiIikuc3VtKCkpLAogICAgICAgICJudW1lcmljX29ubHkiOiBpbnQoZGZbInRleHRfbm9ybSJdLnN0ci5mdWxsbWF0Y2gociJcZCsiKS5zdW0oKSksCiAgICAgICAgInNpbmdsZV9sZXR0ZXJfbm9uX2kiOiBpbnQoKChkZlsidGV4dF9ub3JtIl0uc3RyLmZ1bGxtYXRjaChyIlthLXpdIikpICYgfmRmWyJ0ZXh0X25vcm0iXS5lcSgiaSIpKS5zdW0oKSksCiAgICAgICAgImJyYWNrZXRlZF9jYXB0aW9uIjogaW50KGRmWyJ0ZXh0X25vcm0iXS5zdHIuZnVsbG1hdGNoKHIiXFtbXlxdXStcXSIpLnN1bSgpKSwKICAgICAgICAicHVuY3R1YXRpb25fb25seSI6IGludChkZlsidGV4dF9ub3JtIl0uc3RyLmZ1bGxtYXRjaChyIlteYS16MC05XSsiKS5zdW0oKSksCiAgICB9CiAgICBkZiA9IGRmLmxvY1t+anVua10uY29weSgpCiAgICBzdGF0c1sidXNhYmxlX3Jvd3MiXSA9IGludChsZW4oZGYpKQogICAgc3RhdHNbImR1cGxpY2F0ZV91aWQiXSA9IGludChkZlsidWlkIl0uZHVwbGljYXRlZCgpLnN1bSgpKQogICAgc3RhdHNbInVuaXF1ZV92aWRlb3MiXSA9IGludChkZlsidmlkZW9faWQiXS5udW5pcXVlKCkpCiAgICBzdGF0c1sidW5pcXVlX3RhcmdldHMiXSA9IGludChkZlsidGV4dF9ub3JtIl0ubnVuaXF1ZSgpKQogICAgc3RhdHNbIm9uZV93b3JkX3Jvd3MiXSA9IGludCgoZGZbIm5fd29yZHMiXSA9PSAxKS5zdW0oKSkKICAgIHN0YXRzWyJ0b29fbG9uZ19yb3dzIl0gPSBpbnQoKGRmWyJuX3dvcmRzIl0gPiBNQVhfVE9LRU5TIC0gMikuc3VtKCkpCiAgICBzdGF0c1sidGFyZ2V0X2xlbmd0aCJdID0gewogICAgICAgICJtZWFuIjogZmxvYXQoZGZbIm5fd29yZHMiXS5tZWFuKCkpLAogICAgICAgICJtZWRpYW4iOiBmbG9hdChkZlsibl93b3JkcyJdLm1lZGlhbigpKSwKICAgICAgICAicDkwIjogZmxvYXQoZGZbIm5fd29yZHMiXS5xdWFudGlsZSgwLjkwKSksCiAgICAgICAgIm1heCI6IGludChkZlsibl93b3JkcyJdLm1heCgpKSwKICAgIH0KICAgIHRvcCA9IGRmWyJ0ZXh0X25vcm0iXS52YWx1ZV9jb3VudHMoKS5oZWFkKDEwKQogICAgc3RhdHNbInRvcF90YXJnZXRzIl0gPSBbeyJ0ZXh0IjogdCwgImNvdW50IjogaW50KGMpfSBmb3IgdCwgYyBpbiB0b3AuaXRlbXMoKV0KICAgIHJldHVybiBkZiwgc3RhdHMKCgpkZWYgX3NpZ25lcl9jb2x1bW4oZGY6IHBkLkRhdGFGcmFtZSkgLT4gc3RyIHwgTm9uZToKICAgICIiIlJldHVybiBhIHNpZ25lci1pZCBjb2x1bW4gaWYgdGhlIENTViBjYXJyaWVzIG9uZSAobmFtZSB2YXJpZXMgYnkgZXhwb3J0KS4iIiIKICAgIGxvd2VyZWQgPSB7c3RyKGMpLmxvd2VyKCk6IGMgZm9yIGMgaW4gZGYuY29sdW1uc30KICAgIGZvciBrZXkgaW4gKCJzaWduZXJfaWQiLCAic2lnbmVyaWQiLCAic2lnbmVyIik6CiAgICAgICAgaWYga2V5IGluIGxvd2VyZWQ6CiAgICAgICAgICAgIHJldHVybiBsb3dlcmVkW2tleV0KICAgIHJldHVybiBOb25lCgoKZGVmIHZpZGVvX3NwbGl0KGRmOiBwZC5EYXRhRnJhbWUsIHNlZWQ6IGludCA9IFNFRUQpIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lLCBwZC5EYXRhRnJhbWUsIGRpY3RdOgogICAgIiIiTGVhay1zYWZlIDgwLzEwLzEwIHNwbGl0IGJ5IHZpZGVvX2lkIChzaHVmZmxlZCB3aXRoIGZpeGVkIHNlZWQpLiIiIgogICAgdmlkZW9faWRzID0gbnAuYXJyYXkoc29ydGVkKGRmWyJ2aWRlb19pZCJdLmRyb3BuYSgpLnVuaXF1ZSgpKSwgZHR5cGU9b2JqZWN0KQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBybmcuc2h1ZmZsZSh2aWRlb19pZHMpCiAgICBuID0gbGVuKHZpZGVvX2lkcykKICAgIHRyYWluX2N1dCwgdmFsX2N1dCA9IGludCgwLjgwICogbiksIGludCgwLjkwICogbikKICAgIHRyYWluX2lkcyA9IHNldCh2aWRlb19pZHNbOnRyYWluX2N1dF0pCiAgICB2YWxfaWRzID0gc2V0KHZpZGVvX2lkc1t0cmFpbl9jdXQ6dmFsX2N1dF0pCiAgICB0ZXN0X2lkcyA9IHNldCh2aWRlb19pZHNbdmFsX2N1dDpdKQogICAgYXNzZXJ0IHRyYWluX2lkcy5pc2Rpc2pvaW50KHZhbF9pZHMpIGFuZCB0cmFpbl9pZHMuaXNkaXNqb2ludCh0ZXN0X2lkcykgYW5kIHZhbF9pZHMuaXNkaXNqb2ludCh0ZXN0X2lkcykKICAgIHRyID0gZGZbZGZbInZpZGVvX2lkIl0uaXNpbih0cmFpbl9pZHMpXS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICB2YSA9IGRmW2RmWyJ2aWRlb19pZCJdLmlzaW4odmFsX2lkcyldLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHRlID0gZGZbZGZbInZpZGVvX2lkIl0uaXNpbih0ZXN0X2lkcyldLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIGFzc2VydCBsZW4odHIpICsgbGVuKHZhKSArIGxlbih0ZSkgPT0gbGVuKGRmKSwgInNwbGl0IG11c3QgYWNjb3VudCBmb3IgZXZlcnkgY2xlYW5lZCByb3ciCiAgICAjIENyb3NzLXNwbGl0IHZpZGVvIGxlYWthZ2UgY2hlY2sgKHJvdy1sZXZlbCwgbm90IGp1c3QgaWQgc2V0cykuCiAgICBhc3NlcnQgc2V0KHRyWyJ2aWRlb19pZCJdKS5pc2Rpc2pvaW50KHNldCh2YVsidmlkZW9faWQiXSkpCiAgICBhc3NlcnQgc2V0KHRyWyJ2aWRlb19pZCJdKS5pc2Rpc2pvaW50KHNldCh0ZVsidmlkZW9faWQiXSkpCiAgICBhc3NlcnQgc2V0KHZhWyJ2aWRlb19pZCJdKS5pc2Rpc2pvaW50KHNldCh0ZVsidmlkZW9faWQiXSkpCiAgICBpbmZvID0gewogICAgICAgICJzZWVkIjogc2VlZCwKICAgICAgICAibl92aWRlb3MiOiBuLAogICAgICAgICJ0cmFpbl9yb3dzIjogbGVuKHRyKSwgInZhbF9yb3dzIjogbGVuKHZhKSwgInRlc3Rfcm93cyI6IGxlbih0ZSksCiAgICAgICAgInRyYWluX3ZpZGVvcyI6IGxlbih0cmFpbl9pZHMpLCAidmFsX3ZpZGVvcyI6IGxlbih2YWxfaWRzKSwgInRlc3RfdmlkZW9zIjogbGVuKHRlc3RfaWRzKSwKICAgIH0KICAgICMgU2lnbmVyLWluZGVwZW5kZW5jZSBjaGVjazogdmlkZW8tbGV2ZWwgc3BsaXRzIGNhbiBzdGlsbCBsZWFrIHNpZ25lcnMgaWYKICAgICMgb25lIHNpZ25lciBzcGFucyB2aWRlb3MuIFJlcG9ydGVkIChub3QgYXNzZXJ0ZWQpOiBzb21lIGNvcnBvcmEgcmV1c2UKICAgICMgc2lnbmVycyBhY3Jvc3MgdmlkZW9zIGJ5IGRlc2lnbjsgdGhlIHRyYWluaW5nIHJlcG9ydCBtdXN0IGRpc2Nsb3NlIGl0LgogICAgc2NvbCA9IF9zaWduZXJfY29sdW1uKGRmKQogICAgaWYgc2NvbCBpcyBub3QgTm9uZToKICAgICAgICBzX3RyLCBzX3ZhLCBzX3RlID0gc2V0KHRyW3Njb2xdLmRyb3BuYSgpKSwgc2V0KHZhW3Njb2xdLmRyb3BuYSgpKSwgc2V0KHRlW3Njb2xdLmRyb3BuYSgpKQogICAgICAgIGluZm9bInNpZ25lcl9jb2x1bW4iXSA9IHNjb2wKICAgICAgICBpbmZvWyJzaWduZXJzIl0gPSB7InRyYWluIjogbGVuKHNfdHIpLCAidmFsIjogbGVuKHNfdmEpLCAidGVzdCI6IGxlbihzX3RlKX0KICAgICAgICBpbmZvWyJzaWduZXJfb3ZlcmxhcCJdID0gewogICAgICAgICAgICAidHJhaW5fdmFsIjogc29ydGVkKHNfdHIgJiBzX3ZhKVs6MjBdLAogICAgICAgICAgICAidHJhaW5fdGVzdCI6IHNvcnRlZChzX3RyICYgc190ZSlbOjIwXSwKICAgICAgICAgICAgInZhbF90ZXN0Ijogc29ydGVkKHNfdmEgJiBzX3RlKVs6MjBdLAogICAgICAgICAgICAidHJhaW5fdmFsX2NvdW50IjogbGVuKHNfdHIgJiBzX3ZhKSwKICAgICAgICAgICAgInRyYWluX3Rlc3RfY291bnQiOiBsZW4oc190ciAmIHNfdGUpLAogICAgICAgICAgICAidmFsX3Rlc3RfY291bnQiOiBsZW4oc192YSAmIHNfdGUpLAogICAgICAgIH0KICAgIGVsc2U6CiAgICAgICAgaW5mb1sic2lnbmVyX2NvbHVtbiJdID0gTm9uZQogICAgICAgIGluZm9bInNpZ25lcl9ub3RlIl0gPSAoIm5vIHNpZ25lciBjb2x1bW4gaW4gQ1NWOiBzaWduZXItaW5kZXBlbmRlbmNlIGFzc3VtZWQgZnJvbSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmlkZW8tbGV2ZWwgc3BsaXQgb25seTsgY29uZmlybSBvbiB0aGUgdHJhaW5pbmcgaG9zdC4iKQogICAgcmV0dXJuIHRyLCB2YSwgdGUsIGluZm8KCgpkZWYgcG9zZV9zcG90X2NoZWNrKHVpZHM6IGxpc3Rbc3RyXSwgcG9zZV9zb3VyY2U6IHN0ciwgc2FtcGxlOiBpbnQsIHNlZWQ6IGludCA9IFNFRUQpIC0+IGRpY3Q6CiAgICAiIiJTcG90LWNoZWNrIHBvc2UgbWVtYmVyczogZGltcywgZnJhbWUgY291bnRzLCBOYU4gcmF0ZXMsIG1pc3NpbmcgZmlsZXMuCgogICAgcG9zZV9zb3VyY2UgaXMgYSBsb2NhbCBkaXJlY3Rvcnkgb2YgPHVpZD4ucG9zZSBmaWxlcyBvciBhIGxvY2FsIC56aXAuCiAgICAoUmVtb3RlIFpJUCBzdHJlYW1pbmcgc3RheXMgaW4gdGhlIG5vdGVib29rOyB0aGUgYXVkaXQgYWNjZXB0cyBhIGxvY2FsIGNvcHkuKQogICAgIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBwb3NlX2Zvcm1hdCBpbXBvcnQgUG9zZSAgIyB0eXBlOiBpZ25vcmUKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICByZXR1cm4geyJza2lwcGVkIjogInBvc2VfZm9ybWF0IG5vdCBpbnN0YWxsZWQifQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBwaWNrZWQgPSBbdWlkc1tpXSBmb3IgaSBpbiBybmcuY2hvaWNlKGxlbih1aWRzKSwgc2l6ZT1taW4oc2FtcGxlLCBsZW4odWlkcykpLCByZXBsYWNlPUZhbHNlKV0KICAgIG91dDogZGljdCA9IHsiY2hlY2tlZCI6IDAsICJtaXNzaW5nIjogW10sICJmcmFtZV9jb3VudHMiOiBbXSwgIm5hbl9yYXRlcyI6IFtdLCAiZGltX21pc21hdGNoZXMiOiBbXX0KICAgIGV4cGVjdGVkX3RvdGFsID0gc3VtKG4gZm9yIF8sIG4gaW4gRVhQRUNURURfQ09NUE9ORU5UUykKICAgIGlmIFBhdGgocG9zZV9zb3VyY2UpLmlzX2RpcigpOgogICAgICAgIGdldHRlciA9IGxhbWJkYSB1aWQ6IChQYXRoKHBvc2Vfc291cmNlKSAvIGYie3VpZH0ucG9zZSIpICAjIG5vcWE6IEU3MzEKICAgICAgICBpc196aXAgPSBGYWxzZQogICAgZWxpZiB6aXBmaWxlLmlzX3ppcGZpbGUocG9zZV9zb3VyY2UpOgogICAgICAgIHpmID0gemlwZmlsZS5aaXBGaWxlKHBvc2Vfc291cmNlKQogICAgICAgIG5hbWVzID0gc2V0KHpmLm5hbWVsaXN0KCkpCiAgICAgICAgaXNfemlwID0gVHJ1ZQogICAgZWxzZToKICAgICAgICByZXR1cm4geyJza2lwcGVkIjogZiJwb3NlIHNvdXJjZSBub3QgZm91bmQ6IHtwb3NlX3NvdXJjZX0ifQogICAgZm9yIHVpZCBpbiBwaWNrZWQ6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBpZiBpc196aXA6CiAgICAgICAgICAgICAgICBrZXkgPSBuZXh0KChrIGZvciBrIGluIChmInt1aWR9LnBvc2UiLCBmIiove3VpZH0ucG9zZSIpIGlmIGsgaW4gbmFtZXMpLCBOb25lKQogICAgICAgICAgICAgICAgaWYga2V5IGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgb3V0WyJtaXNzaW5nIl0uYXBwZW5kKHVpZCkKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgd2l0aCB6Zi5vcGVuKGtleSkgYXMgZmg6CiAgICAgICAgICAgICAgICAgICAgcG9zZSA9IFBvc2UucmVhZChmaC5yZWFkKCkpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBwYXRoID0gZ2V0dGVyKHVpZCkKICAgICAgICAgICAgICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgICAgICBvdXRbIm1pc3NpbmciXS5hcHBlbmQodWlkKQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZmg6CiAgICAgICAgICAgICAgICAgICAgcG9zZSA9IFBvc2UucmVhZChmaC5yZWFkKCkpCiAgICAgICAgICAgIGRhdGEgPSBwb3NlLmJvZHkuZGF0YSAgIyBmcmFtZXMgeCBwZW9wbGUgeCBwb2ludHMgeCBkaW1zIChtYXNrZWQgYXJyYXkpCiAgICAgICAgICAgIHRvdGFsX3BvaW50cyA9IGludChkYXRhLnNoYXBlWzJdKQogICAgICAgICAgICBvdXRbImZyYW1lX2NvdW50cyJdLmFwcGVuZChpbnQoZGF0YS5zaGFwZVswXSkpCiAgICAgICAgICAgIG91dFsibmFuX3JhdGVzIl0uYXBwZW5kKGZsb2F0KG5wLmlzbmFuKG5wLmFzYXJyYXkoZGF0YS5kYXRhLCBkdHlwZT1ucC5mbG9hdDY0KSkubWVhbigpKSkKICAgICAgICAgICAgb3V0WyJjaGVja2VkIl0gKz0gMQogICAgICAgICAgICBpZiB0b3RhbF9wb2ludHMgIT0gZXhwZWN0ZWRfdG90YWw6CiAgICAgICAgICAgICAgICBvdXRbImRpbV9taXNtYXRjaGVzIl0uYXBwZW5kKHsidWlkIjogdWlkLCAicG9pbnRzIjogdG90YWxfcG9pbnRzfSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIG91dFsibWlzc2luZyJdLmFwcGVuZChmInt1aWR9IChyZWFkIGVycm9yOiB7ZXhjfSkiKQogICAgaWYgaXNfemlwOgogICAgICAgIHpmLmNsb3NlKCkKICAgIGZjID0gbnAuYXNhcnJheShvdXRbImZyYW1lX2NvdW50cyJdLCBkdHlwZT1mbG9hdCkgaWYgb3V0WyJmcmFtZV9jb3VudHMiXSBlbHNlIG5wLnplcm9zKDApCiAgICBvdXRbImZyYW1lX3N0YXRzIl0gPSAoCiAgICAgICAgeyJtZWFuIjogZmxvYXQoZmMubWVhbigpKSwgIm1lZGlhbiI6IGZsb2F0KG5wLm1lZGlhbihmYykpLCAicDkwIjogZmxvYXQobnAucXVhbnRpbGUoZmMsIDAuOTApKSwKICAgICAgICAgIm1heCI6IGludChmYy5tYXgoKSksICJzYXR1cmF0ZV8xNjAiOiBpbnQoKGZjID49IDE2MCkuc3VtKCkpfSBpZiBsZW4oZmMpIGVsc2Uge30KICAgICkKICAgIG91dFsibWVhbl9uYW5fcmF0ZSJdID0gZmxvYXQobnAubWVhbihvdXRbIm5hbl9yYXRlcyJdKSkgaWYgb3V0WyJuYW5fcmF0ZXMiXSBlbHNlIDAuMAogICAgcmV0dXJuIG91dAoKCmRlZiBydW5fYXVkaXQoY3N2X3BhdGg6IHN0ciwgcG9zZV9zb3VyY2U6IHN0ciB8IE5vbmUgPSBOb25lLCBzYW1wbGU6IGludCA9IDIwMCwKICAgICAgICAgICAgICBzZWVkOiBpbnQgPSBTRUVEKSAtPiBkaWN0OgogICAgZGYgPSBwZC5yZWFkX2Nzdihjc3ZfcGF0aCkKICAgIGlmIHsidWlkIiwgInRleHQifSAtIHNldChkZi5jb2x1bW5zKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJDU1YgbWlzc2luZyByZXF1aXJlZCBjb2x1bW5zOiB7c29ydGVkKHsndWlkJywgJ3RleHQnfSAtIHNldChkZi5jb2x1bW5zKSl9IikKICAgIGNsZWFuZWQsIHN0YXRzID0gY2xlYW5fZGF0YWZyYW1lKGRmKQogICAgaWYgc3RhdHNbImR1cGxpY2F0ZV91aWQiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7c3RhdHNbJ2R1cGxpY2F0ZV91aWQnXX0gZHVwbGljYXRlIFVJRHMgYWZ0ZXIgY2xlYW5pbmcgLS0gcmVmdXNpbmcuIikKICAgIHRyLCB2YSwgdGUsIHNwbGl0ID0gdmlkZW9fc3BsaXQoY2xlYW5lZCwgc2VlZCkKICAgIHJlcG9ydCA9IHsiY3N2Ijogc3RyKGNzdl9wYXRoKSwgImNsZWFuaW5nIjogc3RhdHMsICJzcGxpdCI6IHNwbGl0fQogICAgaWYgcG9zZV9zb3VyY2U6CiAgICAgICAgcmVwb3J0WyJwb3NlX3Nwb3RfY2hlY2siXSA9IHBvc2Vfc3BvdF9jaGVjayhjbGVhbmVkWyJ1aWQiXS50b2xpc3QoKSwgcG9zZV9zb3VyY2UsIHNhbXBsZSwgc2VlZCkKICAgIHJldHVybiByZXBvcnQKCgpkZWYgc2VsZl90ZXN0KCkgLT4gaW50OgogICAgIiIiVmFsaWRhdGUgdGhlIGF1ZGl0IGxvZ2ljIG9uIHN5bnRoZXRpYyBkYXRhIChubyBkYXRhc2V0IG5lZWRlZCkuIiIiCiAgICByb3dzID0gW10KICAgIGZvciB2IGluIHJhbmdlKDIwKToKICAgICAgICBmb3IgcyBpbiByYW5nZSg1KToKICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJ1aWQiOiBmInZpZHt2OjAzZH0te3N9IiwgInRleHQiOiBmIlNhbXBsZSBzZW50ZW5jZSBudW1iZXIge3N9IGZvciB2aWRlbyB7dn0ifSkKICAgIHJvd3MgKz0gWwogICAgICAgIHsidWlkIjogImJhZHVpZCIsICJ0ZXh0IjogIm5vIHNlcXVlbmNlIHBhcnQifSwgICAgICAgICAgIyBtYWxmb3JtZWQgVUlECiAgICAgICAgeyJ1aWQiOiAidmlkMDk5LTAiLCAidGV4dCI6ICIifSwgICAgICAgICAgICAgICAgICAgICAgICAgIyBlbXB0eQogICAgICAgIHsidWlkIjogInZpZDA5OS0xIiwgInRleHQiOiAiMTIzNDUifSwgICAgICAgICAgICAgICAgICAgICMgbnVtZXJpYy1vbmx5CiAgICAgICAgeyJ1aWQiOiAidmlkMDk5LTIiLCAidGV4dCI6ICJ4In0sICAgICAgICAgICAgICAgICAgICAgICAgIyBzaW5nbGUgbGV0dGVyCiAgICAgICAgeyJ1aWQiOiAidmlkMDk5LTMiLCAidGV4dCI6ICJbbXVzaWNdIn0sICAgICAgICAgICAgICAgICAgIyBicmFja2V0ZWQKICAgICAgICB7InVpZCI6ICJ2aWQwOTktNCIsICJ0ZXh0IjogImkifSwgICAgICAgICAgICAgICAgICAgICAgICAjIGxlZ2l0aW1hdGUgcHJvbm91bjoga2VwdAogICAgXQogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXA6CiAgICAgICAgY3N2X3BhdGggPSBzdHIoUGF0aCh0bXApIC8gIm1pbmkuY3N2IikKICAgICAgICBwZC5EYXRhRnJhbWUocm93cykudG9fY3N2KGNzdl9wYXRoLCBpbmRleD1GYWxzZSkKICAgICAgICByZXBvcnQgPSBydW5fYXVkaXQoY3N2X3BhdGgpCiAgICBjLCBzID0gcmVwb3J0WyJjbGVhbmluZyJdLCByZXBvcnRbInNwbGl0Il0KICAgIGFzc2VydCBjWyJyYXdfcm93cyJdID09IDEwNiwgYwogICAgYXNzZXJ0IGNbIm1hbGZvcm1lZF91aWQiXSA9PSAxLCBjCiAgICBhc3NlcnQgY1sianVua19yb3dzIl0gPT0gNCwgYwogICAgYXNzZXJ0IGNbInVzYWJsZV9yb3dzIl0gPT0gMTAxLCBjICAjIGluY2wuIHRoZSBsZWdpdGltYXRlIHByb25vdW4gImkiCiAgICBhc3NlcnQgY1siZHVwbGljYXRlX3VpZCJdID09IDAsIGMKICAgIGFzc2VydCBzWyJ0cmFpbl9yb3dzIl0gKyBzWyJ2YWxfcm93cyJdICsgc1sidGVzdF9yb3dzIl0gPT0gMTAxLCBzCiAgICBhc3NlcnQgc1sibl92aWRlb3MiXSA9PSAyMSwgcyAgIyAyMCBmdWxsIHZpZGVvcyArIHZpZDA5OSAoa2VwdCB2aWEgImkiKQogICAgYXNzZXJ0IHNbInNpZ25lcl9jb2x1bW4iXSBpcyBOb25lIGFuZCAic2lnbmVyX25vdGUiIGluIHMsIHMKICAgICMgU2lnbmVyLWNvbHVtbiBwYXRoOiBzYW1lIHNpZ25lciBhY3Jvc3MgdmlkZW9zIG11c3QgYmUgcmVwb3J0ZWQuCiAgICBkZjIgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgW3sidWlkIjogZiJ2e3Z9LTAiLCAidmlkZW9faWQiOiBmInZ7dn0iLCAidGV4dCI6ICJoZWxsbyB3b3JsZCIsCiAgICAgICAgICAic2lnbmVyIjogImFsaWNlIiBpZiB2IDwgMiBlbHNlICJib2IifQogICAgICAgICBmb3IgdiBpbiByYW5nZSg0KV0KICAgICkKICAgIF8sIF8sIF8sIHMyID0gdmlkZW9fc3BsaXQoZGYyLCBzZWVkPTEpCiAgICBhc3NlcnQgczJbInNpZ25lcl9jb2x1bW4iXSA9PSAic2lnbmVyIiwgczIKICAgIGFzc2VydCBzMlsic2lnbmVyX292ZXJsYXAiXVsidHJhaW5fdmFsX2NvdW50Il0gKyBzMlsic2lnbmVyX292ZXJsYXAiXVsidHJhaW5fdGVzdF9jb3VudCJdIFwKICAgICAgICArIHMyWyJzaWduZXJfb3ZlcmxhcCJdWyJ2YWxfdGVzdF9jb3VudCJdID49IDAsIHMyCiAgICBwcmludChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTEpWzoxNTAwXSkKICAgIHByaW50KCJTRUxGLVRFU1QgUEFTUzogY2xlYW5pbmcgY291bnRzLCBwcm9ub3VuIHJldGVudGlvbiwgbGVhay1zYWZlIHNwbGl0IGFsbCBhcyBleHBlY3RlZC4iKQogICAgcmV0dXJuIDAKCgpkZWYgbWFpbihhcmd2OiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSkgLT4gaW50OgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iU2lnblNhZ2UgaVNpZ24gZGF0YXNldCBpbnRlZ3JpdHkgYXVkaXQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNzdiIsIGhlbHA9InBhdGggdG8gaVNpZ25fdjEuMS5jc3YiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBvc2Utc291cmNlIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJsb2NhbCAucG9zZSBkaXIgb3IgLnppcCAob3B0aW9uYWwpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zYW1wbGUiLCB0eXBlPWludCwgZGVmYXVsdD0yMDApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PVNFRUQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJ3cml0ZSBKU09OIHJlcG9ydCBoZXJlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWxmLXRlc3QiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoYXJndikKICAgIGlmIGFyZ3Muc2VsZl90ZXN0OgogICAgICAgIHJldHVybiBzZWxmX3Rlc3QoKQogICAgaWYgbm90IGFyZ3MuY3N2OgogICAgICAgIGFwLmVycm9yKCItLWNzdiBpcyByZXF1aXJlZCAob3IgdXNlIC0tc2VsZi10ZXN0KSIpCiAgICByZXBvcnQgPSBydW5fYXVkaXQoYXJncy5jc3YsIGFyZ3MucG9zZV9zb3VyY2UsIGFyZ3Muc2FtcGxlLCBhcmdzLnNlZWQpCiAgICB0ZXh0ID0ganNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0xKQogICAgaWYgYXJncy5vdXQ6CiAgICAgICAgUGF0aChhcmdzLm91dCkud3JpdGVfdGV4dCh0ZXh0KQogICAgICAgIHByaW50KGYicmVwb3J0IHdyaXR0ZW4gdG8ge2FyZ3Mub3V0fSIpCiAgICBwcmludCh0ZXh0WzozMDAwXSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK"}, {"path": "research/training/train_sign_to_text.py", "root": "SUP", "sha256": "1938fefcdff4bba5793784483acf2c17e166c3086d2efd75cb242c30381f8194", "b64": "..."}]
for _item in BUNDLED:
    _base = POC if _item["root"] == "POC" else SUP
    _p = _base / _item["path"]
    _data = _b64.b64decode(_item["b64"])
    assert _hl.sha256(_data).hexdigest() == _item["sha256"], \
        f"bundled blob corrupt: {_item['path']}"
    if _p.is_file() and _hl.sha256(_p.read_bytes()).hexdigest() == _item["sha256"]:
        continue  # already present and byte-identical
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(_data)
    print(f"wrote {_p}")
print("SignSage T5 POC bootstrap")
print("=========================")
print(f"POC directory: {POC}")
print(f"Support root:  {SUP}")
print()
_ok = True
for _name in ("model.py", "materialize.py", "train.py", "evaluate.py"):
    _good = (POC / _name).is_file()
    print(f"{_name:16s} {'OK' if _good else 'MISSING'}")
    _ok &= _good
if not _ok:
    raise RuntimeError("bootstrap failed: POC source files missing after restore")
import py_compile as _pc
for _name in ("model.py", "materialize.py", "train.py", "evaluate.py"):
    _pc.compile(str(POC / _name), doraise=True)
print("py_compile: all OK")
print(f"model: {POC / 'model.py'}")
print(f"materialize: {POC / 'materialize.py'}")
print(f"train: {POC / 'train.py'}")
print(f"evaluate: {POC / 'evaluate.py'}")


## PHASE 0 — environment / setup (pip + GPU report + import check)


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
import subprocess as _sp
_sp.run(["pip", "install", "-q", "transformers", "huggingface_hub",
         "pose-format", "jiwer", "sacrebleu", "rouge_score", "nltk"], check=True)
import torch as _t
print(f"torch={_t.__version__} cuda={_t.cuda.is_available()}")
if _t.cuda.is_available():
    print(f"gpu={_t.cuda.get_device_name(0)} count={_t.cuda.device_count()}")
    _free, _total = _t.cuda.mem_get_info(0)
    print(f"vram free={_free / 1e9:.1f}G total={_total / 1e9:.1f}G")
import transformers as _tr
print(f"transformers={_tr.__version__}")


## BOOTSTRAP VERIFICATION — support files exist AND everything imports (fails loud)


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
print(f"POC: {POC}")
print("Required support files:")
_REQ = {
    "audit_dataset.py": SUP / "research" / "training" / "audit_dataset.py",
    "train_sign_to_text.py": SUP / "research" / "training" / "train_sign_to_text.py",
    "app/ml/preprocessing/sequence.py": SUP / "backend" / "app" / "ml" / "preprocessing" / "sequence.py",
    "app/ml/models/sign_to_text.py": SUP / "backend" / "app" / "ml" / "models" / "sign_to_text.py",
}
_ok = True
for _name, _p in _REQ.items():
    _good = _p.is_file()
    print(f"  {_name} -> {_p} {'EXISTS' if _good else 'MISSING'}")
    _ok = _ok and _good
if not _ok:
    print("sys.path searched:")
    import sys as _s2
    for _e in _s2.path:
        print(f"  {_e}")
    for _d in (SUP / "research" / "training", SUP / "backend"):
        print(f"{_d}: " + (str(sorted(p.name for p in _d.iterdir())) if _d.is_dir() else "ABSENT"))
    raise RuntimeError("support tree incomplete; refusing to continue")
import sys as _s
_s.path.insert(0, str(POC))
_s.path.insert(0, str(SUP / "backend"))
_s.path.insert(0, str(SUP / "research" / "training"))
try:
    import audit_dataset as _ad, train_sign_to_text as _tst  # noqa
    import model as _m, materialize as _mat, train as _trn, evaluate as _ev  # noqa
except ModuleNotFoundError as _e:
    print(f"import failed: {_e}; sys.path=")
    for _e2 in _s.path:
        print(f"  {_e2}")
    raise
print("all POC + support imports: OK")


## PHASE 1 — materialization smoke / resume verification (no dataset, no token, no GPU)


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
import subprocess, sys as _sys
_PY = _sys.executable
subprocess.run([_PY, str(POC / "materialize.py"), "--smoke",
                "--out", str(OUT / "smoke"), "--shard-size", "10",
                "--sanity"], check=True)
# Resume: second run must skip everything and stay valid.
subprocess.run([_PY, str(POC / "materialize.py"), "--smoke",
                "--out", str(OUT / "smoke"), "--shard-size", "10"], check=True)
# Interruption recovery: delete a shard, re-run, verify restore.
import json as _j, hashlib as _h
_train_shards = sorted((OUT / "smoke").glob("train_*.npz"))
assert _train_shards, "no train shards after smoke"
_train_shards[1].unlink()
subprocess.run([_PY, str(POC / "materialize.py"), "--smoke",
                "--out", str(OUT / "smoke"), "--shard-size", "10"], check=True)
_exp = {"train": (48, 5), "val": (6, 1), "test": (6, 1)}
_total_shards = 0
for _name, (_n, _s) in _exp.items():
    _m = _j.loads((OUT / "smoke" / f"{_name}_manifest.json").read_text())
    assert _m["n"] == _n, (_name, _m["n"])
    assert len(_m["shards"]) == _s, (_name, len(_m["shards"]))
    assert _m["skip_rate"] == 0.0, (_name, _m["skip_rate"])
    for _sh in _m["shards"]:
        _p = OUT / "smoke" / _sh["file"]
        assert _h.sha256(_p.read_bytes()).hexdigest() == _sh["sha256"], _sh["file"]
    _total_shards += len(_m["shards"])
assert _total_shards == 7, _total_shards
print("PHASE 1 OK: 48/6/6 rows, 7 shards, skip_rate=0, resume+recovery verified")
set_state("phase1", "ok")


## PHASE 2 — real-data materialization (~5k, one streaming pass; re-run to resume)


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
need("phase1")
import os as _os, subprocess, sys as _sys
import torch as _t
if not _t.cuda.is_available():
    raise RuntimeError("no GPU: refusing a multi-hour CPU materialization")
try:
    from kaggle_secrets import UserSecretsClient as _USC
    _HF_TOKEN = _USC().get_secret("HF_TOKEN") or None
except Exception:
    _HF_TOKEN = None  # not on Kaggle or secret not attached
if not _HF_TOKEN:
    _HF_TOKEN = _os.environ.get("HF_TOKEN") or None  # plain-env fallback
if not _HF_TOKEN:
    raise RuntimeError("HF_TOKEN secret unavailable. "
                         "Add/attach Kaggle Secret named HF_TOKEN.")
print("HF_TOKEN configured: YES")
if not CSV or not Path(CSV).is_file():
    raise RuntimeError(f"CSV input missing: {CSV!r}; attach iSign CSV as a dataset input")
_ENV = _os.environ.copy()
_ENV["HF_TOKEN"] = _HF_TOKEN  # subprocess-only; notebook env untouched
subprocess.run([_sys.executable, str(POC / "materialize.py"),
                "--csv", CSV, "--pose-source", "remote",
                "--out", str(OUT / "shards"),
                "--target-train", "4000", "--target-val", "500",
                "--target-test", "500", "--sanity"], check=True, env=_ENV)
import json as _j
_rep = _j.loads((OUT / "shards" / "materialize_report.json").read_text())
for _name in ("train", "val", "test"):
    _s = _rep["subset"][_name]
    assert _s["n"] > 0 and _s["skip_rate"] <= 0.05, (_name, _s)
    print(_name, "n=", _s["n"], "shards=", len(_s["shards"]),
          "skip_rate=", round(_s["skip_rate"], 4))
print("PHASE 2 OK")
set_state("phase2", "ok")


## PHASE 3 — real 64-sample overfit gate (BINDING GO/NO-GO).
FAIL (<95% EM) stops the notebook here; Phase 4 will refuse to run.


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
need("phase2")
import subprocess, sys as _sys
import torch as _t
if not _t.cuda.is_available():
    raise RuntimeError("no GPU: gate belongs on GPU")
subprocess.run([_sys.executable, str(POC / "train.py"), "--real-gate",
                "--shards", str(OUT / "shards"),
                "--out", str(OUT / "gate64"), "--gate-n", "64"], check=True)
import json as _j
_g = _j.loads((OUT / "gate64" / "gate_report.json").read_text())
print(f"EM stage A={_g['stage_a']['exact_match']:.1%} "
      f"stage B={_g['stage_b']['exact_match']:.1%} passed={_g['passed']}")
if not _g["passed"]:
    raise RuntimeError("GATE FAILED: EM < 95%. STOP — diagnose, do not run Phase 4.")
print("PHASE 3 PASSED")
set_state("phase3", "ok")


## PHASE 4 — 5k experiment (ONLY if Phase 3 passed; staged A->B, val-tracked, resumable)


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
need("phase3")
import json as _j, subprocess, sys as _sys
_g = _j.loads((OUT / "gate64" / "gate_report.json").read_text())
if not _g.get("passed"):
    raise RuntimeError("gate did not pass; Phase 4 refused")
subprocess.run([_sys.executable, str(POC / "train.py"),
                "--shards", str(OUT / "shards"),
                "--out", str(OUT / "exp5k"),
                "--epochs", "8", "--batch", "16", "--patience", "3"], check=True)
# After preemption re-run with: train.py ... --resume (same command + flag)
assert (OUT / "exp5k" / "best.pt").is_file(), "best checkpoint missing"
print("PHASE 4 OK")
set_state("phase4", "ok")


## PHASE 5 — evaluation (WER / BLEU-1..4 / ROUGE-L / EM / diversity / baseline / sensitivity)


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
need("phase4")
import subprocess, sys as _sys
assert (OUT / "exp5k" / "best.pt").is_file(), "best checkpoint missing"
subprocess.run([_sys.executable, str(POC / "evaluate.py"),
                "--checkpoint", str(OUT / "exp5k" / "best.pt"),
                "--shards", str(OUT / "shards"),
                "--out", str(OUT / "eval5k")], check=True)
print("PHASE 5 OK")
set_state("phase5", "ok")


## PHASE 6 — artifact persistence (`/kaggle/working` is ephemeral: publish everything as a versioned dataset).


In [ ]:
assert EXECUTE, "set EXECUTE = True to run this phase"
need("phase5")
import json as _j, shutil as _sh
_rep = OUT / "reports"; _rep.mkdir(exist_ok=True)
for _f in ["state.json"]:
    _sh.copy(OUT / _f, _rep / _f)
for _d, _f in [("gate64", "gate_report.json"), ("exp5k", "train_report.json"),
                ("eval5k", "eval_report.json"), ("shards", "materialize_report.json")]:
    _src = OUT / _d / _f
    assert _src.is_file(), f"missing {_src}"
    _sh.copy(_src, _rep / f"{_d}_{_f}")
_total = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())
print(f"OUT tree: {OUT} total={_total / 1e9:.2f} GB")
for _d in ("smoke", "shards", "gate64", "exp5k", "eval5k", "reports"):
    _p = OUT / _d
    if _p.is_dir():
        _n = sum(1 for _ in _p.rglob('*') if _.is_file())
        print(f"  {_d}/: {_n} files")
print("Publish as a versioned Kaggle dataset (UI: Output -> ... or kaggle API):")
print("  shards/*.npz + *_manifest.json + materialize_report.json")
print("  exp5k/best.pt + exp5k/train_report.json")
print("  gate64/gate_report.json + eval5k/eval_report.json")
print("PHASE 6 OK")
set_state("phase6", "ok")
